# Workit RAG 평가(eval) on Colab (GPU) — 단계적(Staged) Sweep, reranker 단일화 버전

이 노트북은 로컬에서 만든 RAG 파이프라인(JoRAG/HoRAG)을 Colab GPU에서
돌리기 위한 것입니다. 로컬 Qdrant 서버에 접속하는 대신, 이미 임베딩이 끝난
파일들을 Colab에 직접 업로드해서 **Colab 세션 안에서 로컬(파일 기반) Qdrant를
새로 만들고** 그 위에서 단계적 sweep을 실행합니다.

## 이전 버전과 달라진 점
- **평가 기준을 조(jo) 단위로 통일**: LLM에는 결국 조 텍스트가 최종
  전달되므로(HoRAG/Cascaded도 parent fetch로 조 텍스트를 붙여서 반환),
  jo/ho/cascaded 세 variant 모두 정답을 `relevant_docs_jo` 기준으로 채점
  (ho/cascaded는 raw 후보를 넉넉히 뽑아 조 단위로 접어서 top-k 계산).
  `eval_set.json`의 쿼리별 `category` 필드(위반/누락/정상)는 평가 로직에서
  쓰이지 않아 제거함.
- **reranker를 `bge-reranker-v2-m3` 하나로 통합** (ko-reranker/reranker1 계열 제거)
- `alpha`, `rrf_k`, `fetch_k`, `rerank_k`, `use_cross_refs`, `max_cross_ref_tokens`를
  **완전 교차곱이 아니라 단계적(staged)으로** sweep — 조합 폭발(2280개까지 감)을 피하기 위함
  - 0단계: reranker OFF baseline
  - 1단계: alpha × rrf_k (거의 공짜인 축 먼저 넓게)
  - 2단계: fetch_k × rerank_k (1단계 최적값 고정, 비용 드는 축만 따로)
  - 3단계 (HoRAG만): use_cross_refs on/off + max_cross_ref_tokens (2단계 최적값 고정)
- `top_k`는 sweep 축이 아니라, 한 번 검색한 결과에서 Recall@1/3/5/10을 전부 계산

## 실행 전 체크리스트
1. **런타임 → 런타임 유형 변경 → GPU(T4)** 로 바꾸세요.
2. 왼쪽 파일 탐색기(📁)에서 `/content/` 에 아래 파일들을 업로드하세요.
   - `vectors_jo.npz`
   - `vectors_ho.npz`
   - `sparse_weights_jo.json`
   - `sparse_weights_ho.json`
   - `chunks_jo.json`
   - `chunks_ho.json`
   - `eval_set.json` (평가셋 — 50개든 100개든 파일명이 다르면 아래 셀에서 경로만 바꿔주세요)
3. 위 파일들이 다 올라간 뒤 셀을 위에서부터 순서대로 실행하세요.

## 노트북 구성
1. 패키지 설치 / GPU 확인
2. 파이프라인 코드 작성 (`yoonha_law_rag.py`, `yoonha_colab_upsert.py`)
3. 업로드 파일 존재 확인
4. 로컬 Qdrant 생성 + upsert
5. 단계적(Staged) sweep 실행 (`yoonha_rag_eval_staged.py`)
6. 결과 확인 + 다운로드


## 1. 패키지 설치 / GPU 확인

In [1]:
!pip install qdrant-client FlagEmbedding transformers torch pandas --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.8 MB/s eta 0:00:00


In [2]:
import torch
print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️  GPU가 안 잡혔습니다. 런타임 → 런타임 유형 변경 → GPU(T4)로 바꾼 뒤 다시 실행하세요.")

CUDA 사용 가능: True
GPU: Tesla T4


In [3]:
!nvidia-smi

Tue Jul 14 03:26:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. 파이프라인 코드 작성

아래 2개 셀은 `%%writefile`로 `/content/`에 실제 `.py` 파일을 만듭니다.
`yoonha_law_rag.py`는 reranker 단일화 + `rrf_k`/`max_cross_ref_tokens` 축이
추가된 개정판입니다.

In [4]:
%%writefile /content/yoonha_law_rag.py
"""
Workit - 계약서 검토 RAG 파이프라인 (2-flow, sweep 가능한 버전 + 캐싱)
파일명: rag/yoonha_law_rag.py

전체 흐름 (다이어그램 기준):
  1) JoRAG   : law_kb_jo 에서 "조" 단위로 직접 검색 (넓은 단위, 청킹 크기 큼)
  2) HoRAG   : law_kb_ho 에서 "호/목/세목"까지 쪼갠 세부 단위로 검색
               → 후보들의 cross_refs로 관련 조항 추가 확장 (별도 xref 컬렉션 없음,
                 cross_refs가 이미 ho payload 안에 들어있음)
               → 항상 parent_chunk_id로 "조" 텍스트를 fetch해서 최종 출력 단위를
                 조로 통일 (하위 단위는 검색에만 쓰고 최종 결과로는 노출 안 함)
  두 flow 모두 계약서 조항 1개당 결과를 병합해서 list[ClauseResult]로 반환.

이번 개정 (2026-07-04, reranker 단일화 + sweep 축 확장):
  - reranker를 bge-reranker-v2-m3 하나로 고정. 기존 reranker1(ko-reranker)/
    reranker2 이원 구조를 전부 걷어내고 reranker 파라미터 하나로 통합했다.
    JoRAG/HoRAG 둘 다 이 reranker 하나만 쓴다.
  - RRF_K가 _hybrid_search 안에 하드코딩(60)돼 있던 걸 함수 인자로 뺐다.
    alpha와 마찬가지로 RRF 합산 단계에서만 쓰이므로, raw_search 캐시를
    전혀 건드리지 않고 sweep 가능하다 (사실상 공짜 축).
  - HoRAG의 cross-ref 확장에 토큰 예산(max_cross_ref_tokens) 개념을 추가했다.
    "몇 개까지 추가할지"가 아니라 "reranker가 실제로 받는 총 토큰이 예산을
    넘지 않는 선까지 추가"하는 방식이다. 원본 후보(hybrid search로 직접 찾은
    것)는 무조건 유지하고, cross-ref로 딸려온 추가 후보만 트리밍 대상이다.
    우선순위는 "어떤 원본 후보가 인용한 것인지"의 rrf_score(source_score)가
    높은 순.
  - 이 트리밍은 parent fetch *이후*에 수행한다. cross-ref 확장 시점(parent
    fetch 전)에는 아직 호/목 단위의 짧은 텍스트라서 토큰 수를 재봤자 의미가
    없다 — parent fetch로 조 전체 텍스트로 바뀐 뒤에야 실제로 reranker가
    보게 될 길이를 알 수 있다.
  - top_k는 기존과 동일하게 "이미 순위 매겨진 후보 리스트를 자르기만" 하는
    구조라 별도 캐싱/재검색이 필요 없다. sweep 스크립트에서는 top_k=1/3/5/10을
    동일한 한 번의 검색 결과에서 전부 계산하면 된다 (재검색 불필요).

sweep 가능하게 바뀐 점 (이전 버전과 공통):
  - alpha, use_reranker, rerank_k, fetch_k, top_k, rrf_k, max_cross_ref_tokens을
    전부 모듈 상수가 아니라 함수 인자로 뺐다.
  - 모듈 상수는 "sweep 안 할 때 쓰는 기본값" 역할만 한다 (DEFAULT_* 접두사).

캐싱 구조:
  - alpha, rrf_k는 RRF 가중치 합산에만 쓰이고, 임베딩/Qdrant raw 검색 결과와는
    무관하다. 그래서 이 두 값을 sweep할 때 (collection, query_text, fetch_k)별
    raw 검색 결과를 한 번만 구하고, 조합마다는 캐시된 raw 결과로 RRF 점수만
    다시 계산한다.
  - reranker 점수는 (reranker_name, query_text, chunk_id) 쌍에만 의존한다.
    alpha/rrf_k/max_cross_ref_tokens이 바뀌면 어떤 chunk_id가 rerank_k 안에
    들어오는지는 달라질 수 있지만, 같은 chunk_id에 대한 점수 자체는 항상 같다.
    그래서 미스(cache miss)만 골라서 계산하고 나머지는 캐시에서 꺼내 쓴다.
  - parent fetch(조 텍스트)와 cross-ref 확장(scroll 조회)도 chunk_id -> payload
    조회라 alpha/rrf_k/쿼리와 무관하게 전역 캐시가 가능하다. 여러 조항이 같은
    법 조항을 참조하는 경우가 많아서 쿼리 간에도 재사용된다.
  - fetch_k는 예외다 — raw_search 캐시 키에 fetch_k가 포함돼 있어서, fetch_k
    값이 바뀌면 Qdrant 검색 자체가 다시 돈다 (alpha/rrf_k처럼 공짜가 아님).
  - cache=None으로 호출하면 캐싱 없이 기존과 완전히 동일하게 동작한다.

컬렉션:
  - JoRAG : law_kb_jo (조 단위, parent 없음)
  - HoRAG : law_kb_ho (호/목/세목 단위 + cross_refs, parent fetch → 조 텍스트)

공통 출력: list[ClauseResult]  ← 항상 조 단위로 반환
"""

from __future__ import annotations

import json
import pickle
import re
from dataclasses import dataclass, field, asdict
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from FlagEmbedding import BGEM3FlagModel
from qdrant_client import QdrantClient
from qdrant_client.models import SparseVector, Filter, FieldCondition, MatchValue

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 경로 / 설정
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
_THIS_DIR     = Path(__file__).resolve().parent
_DATA_DIR     = _THIS_DIR.parent / "data"
LAWS_REF_PATH = _DATA_DIR / "hn_seed" / "law_refs.json"

QDRANT_HOST = "localhost"
QDRANT_PORT = 6333

COLLECTION_JO = "law_kb_jo"
COLLECTION_HO = "law_kb_ho"
# HoXrefRAG는 별도 컬렉션 없이 COLLECTION_HO를 그대로 씀 (cross_refs가 payload에 이미 있음)

EMBED_MODEL = "BAAI/bge-m3"

# sweep 안 할 때 쓰는 기본값. 실제 최적값은 yoonha_rag_eval.py로 찾는다.
DEFAULT_FETCH_K              = 50
DEFAULT_RERANK_K             = 10
DEFAULT_TOP_K                = 10
DEFAULT_ALPHA                = 0.5    # 1.0 = dense only, 0.0 = sparse only
DEFAULT_RRF_K                = 60     # RRF 공식의 스무딩 상수 (기존 하드코딩값)
DEFAULT_MAX_CROSS_REF_TOKENS = None   # None = 트리밍 없음 (기존과 동일 동작)

# reranker는 이제 이거 하나만 쓴다 (ko-reranker/reranker1 계열 제거).
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 확정된 최적 조합
# reranker가 하나로 통합됐으므로, use_reranker on/off만 남는다.
# fetch_k / rerank_k / rrf_k / max_cross_ref_tokens의 최종값은
# yoonha_rag_eval.py 재sweep 결과로 다시 채워 넣을 것.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

BEST_JO_CONFIG = dict(
    use_reranker=True,
    alpha=DEFAULT_ALPHA,
)

BEST_HO_CONFIG = dict(
    use_reranker=True,
    alpha=DEFAULT_ALPHA,
    use_cross_refs=True,
    max_cross_ref_tokens=DEFAULT_MAX_CROSS_REF_TOKENS,
)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Sweep 캐시
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

@dataclass
class SweepCache:
    """
    alpha × rrf_k × reranker on/off × fetch_k × ... sweep에서 재사용 가능한
    중간 계산 결과를 담는 캐시.

    - embed        : query_text -> (dense_vec, sparse_vec)
                      (임베딩은 collection/alpha/rrf_k와 무관 — jo/ho variant 간에도 공유됨)
    - raw_search   : (collection, query_text, fetch_k) -> (dense_points, sparse_points)
                      (alpha/rrf_k와 무관한 Qdrant 원본 검색 결과. RRF 합산만 alpha/rrf_k
                       조합마다 다시 함. fetch_k가 바뀌면 캐시 키 자체가 달라짐 — 재검색 필요.)
    - scroll       : (collection, chunk_id) -> payload
                      (parent fetch / cross-ref 확장에서 쓰는 단건 조회. 여러 쿼리에서
                       같은 법 조항을 참조하면 자동으로 재사용됨)
    - rerank_score : (reranker_name, query_text, chunk_id) -> score
                      (reranker cross-encoder 점수. alpha/rrf_k/max_cross_ref_tokens이
                       달라져도 같은 chunk_id에 대한 점수는 항상 같으므로 미스만 계산)

    cache=None으로 함수를 호출하면 캐싱 없이 기존과 동일하게 동작한다.
    """
    embed        : dict = field(default_factory=dict)
    raw_search   : dict = field(default_factory=dict)
    scroll       : dict = field(default_factory=dict)
    rerank_score : dict = field(default_factory=dict)

    def stats(self) -> dict:
        return {
            "embed_cached":      len(self.embed),
            "raw_search_cached": len(self.raw_search),
            "scroll_cached":     len(self.scroll),
            "rerank_cached":     len(self.rerank_score),
        }

    def save(self, path: str | Path) -> None:
        """캐시를 디스크에 pickle로 저장한다. Colab 세션이 끊겨도 다음 실행에서
        load()로 이어받을 수 있게 하기 위함 — 세션마다 캐시가 초기화되면서
        sweep 전체를 처음부터 다시 도는 게 체감 속도 저하의 큰 원인이었다."""
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        tmp_path = path.with_suffix(path.suffix + ".tmp")
        with open(tmp_path, "wb") as f:
            pickle.dump(self, f, protocol=pickle.HIGHEST_PROTOCOL)
        tmp_path.replace(path)  # 원자적 교체 — 저장 도중 끊겨도 기존 파일은 안전
        print(f"  💾 캐시 저장: {path} ({self.stats()})")

    @classmethod
    def load_or_new(cls, path: str | Path) -> "SweepCache":
        """path에 저장된 캐시가 있으면 불러오고, 없으면 빈 캐시로 새로 시작한다."""
        path = Path(path)
        if path.exists():
            with open(path, "rb") as f:
                cache = pickle.load(f)
            print(f"  💾 캐시 불러옴: {path} ({cache.stats()})")
            return cache
        print(f"  💾 캐시 파일 없음, 새로 시작: {path}")
        return cls()


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Cross-encoder Reranker
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class CrossEncoderReranker:
    """
    transformers AutoModel 기반 Cross-encoder reranker.
    FlagReranker 대체용 — 최신 transformers 호환.
    """

    def __init__(self, model_name: str, device: str = "cpu"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model     = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.model.to(device)
        self.model.eval()
        self.device = device

    def compute_score(
        self,
        pairs     : list[list[str]],
        batch_size: int  = 32,
        normalize : bool = True,
    ) -> list[float]:
        all_scores: list[float] = []

        for i in range(0, len(pairs), batch_size):
            batch   = pairs[i : i + batch_size]
            encoded = self.tokenizer(
                [p[0] for p in batch],
                [p[1] for p in batch],
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt",
            )
            encoded = {k: v.to(self.device) for k, v in encoded.items()}

            with torch.no_grad():
                logits = self.model(**encoded).logits

            scores = logits.squeeze(-1) if logits.shape[-1] == 1 else logits[:, 1]
            if normalize:
                scores = torch.sigmoid(scores)

            all_scores.extend(scores.cpu().tolist())

        return all_scores

    def count_tokens(self, text: str) -> int:
        """토큰 예산 계산용. truncation 없이 실제 토큰 수를 그대로 센다."""
        return len(self.tokenizer.encode(text, add_special_tokens=False))


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 데이터 클래스
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

@dataclass
class LawRef:
    """검색된 법령 조문 1건."""
    chunk_id   : str
    article    : str
    category   : str
    law_name   : str
    chunk_text : str
    score      : float
    is_risk_ref: bool
    parent_id  : str = ""
    cross_refs : list[str] = field(default_factory=list)  # HoRAG(xref 확장) 전용


@dataclass
class ClauseResult:
    """계약서 조항 1건의 검색 결과 — 항상 조 단위."""
    clause_number: str
    clause_text  : str
    page         : int            = 0
    bbox         : dict | None    = None
    law_refs     : list[LawRef]   = field(default_factory=list)
    categories   : list[str]      = field(default_factory=list)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 공통 유틸
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def load_laws_ref(path: Path = LAWS_REF_PATH) -> dict[str, dict]:
    if not path.exists():
        print(f"  ⚠️  laws_ref.json 없음: {path}")
        return {}
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def load_embed_model(model_name: str = EMBED_MODEL, use_fp16: bool = True) -> BGEM3FlagModel:
    print(f"📦 임베딩 모델 로드: {model_name}")
    return BGEM3FlagModel(model_name, use_fp16=use_fp16)


def load_reranker(device: str = "cpu") -> CrossEncoderReranker:
    """
    reranker는 로드 비용이 커서 sweep 중에는 한 번만 로드해두고,
    실제로 쓸지 말지는 각 검색 함수의 use_reranker 플래그로 토글한다.
    """
    print(f"📦 Re-ranker 로드: {RERANKER_MODEL}")
    return CrossEncoderReranker(RERANKER_MODEL, device=device)


def derive_jo_id(chunk_id: str) -> str:
    """
    ho-level chunk_id 문자열에서 그 조에 해당하는 jo-level chunk_id를 직접 역산한다.

    ho id 형식: {prefix}_{장}_{절}_{조}_{항}_{호}_{목}_{세목} (8토큰 고정)
    jo id 형식: 일반 법령은 {prefix}_{장}_{절}_{조} (앞 4토큰),
                PYG(예규)는 조가 없어 항이 anchor이므로 {prefix}_{장}_{절}_{조=0}_{항} (앞 5토큰).

    주의: payload의 parent_chunk_id 필드는 이 용도로 쓰면 안 된다. 실제 데이터를
    검증해보면 parent_chunk_id는 JO 컬렉션이 아니라 HO 컬렉션 자기 자신 안의 다른
    chunk(조 단위로 롤업된 chunk, 혹은 중간 단계인 호/목)를 가리키고 있어서 JO
    컬렉션 chunk_id와 절대 일치하지 않는다 (검증: ho 7640개 중 parent_chunk_id가
    JO chunk_id와 일치한 건 0개). 반면 이 함수처럼 chunk_id 자신의 앞쪽 토큰만
    잘라내는 방식은 ho 7640개 전부 100% 올바른 jo_id로 매핑된다 (leaf 청크든
    조 단위 롤업 청크든 동일하게 성립).
    """
    tokens = chunk_id.split("_")
    if tokens[0] == "PYG":
        return "_".join(tokens[:5])
    return "_".join(tokens[:4])


def get_vectors(
    text : str,
    model: BGEM3FlagModel,
    cache: SweepCache | None = None,
) -> tuple[list[float], dict[int, float]]:
    """
    BGE-M3 dense/sparse 임베딩. collection이나 alpha/rrf_k와 무관하므로
    query_text만으로 캐시 가능 (jo/ho variant 간에도 재사용됨).
    """
    if cache is not None and text in cache.embed:
        return cache.embed[text]

    output = model.encode(
        [text],
        return_dense=True,
        return_sparse=True,
        return_colbert_vecs=False,
    )
    dense_vec       = output["dense_vecs"][0].tolist()
    lexical_weights = output["lexical_weights"][0]

    sparse_vec: dict[int, float] = {}
    for token_str, weight in lexical_weights.items():
        token_id = model.tokenizer.convert_tokens_to_ids(token_str)
        if isinstance(token_id, int):
            sparse_vec[token_id] = sparse_vec.get(token_id, 0.0) + float(weight)

    if cache is not None:
        cache.embed[text] = (dense_vec, sparse_vec)

    return dense_vec, sparse_vec


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 계약서 청킹 (조 단위 출력)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def chunk_contract(text: str) -> list[dict]:
    """계약서를 조 단위로 청킹."""
    HANG_MAP = {c: i + 1 for i, c in enumerate("①②③④⑤⑥⑦⑧⑨⑩⑪⑫⑬⑭⑮")}
    HO_SPLIT_PATTERN = r"(?:^|\s)(\d{1,2}\.\s)"

    text = text.strip()
    header_pattern = re.compile(r"제(\d+)조(?:의(\d+))?\s*\(([^)]*)\)")
    raw_matches    = list(header_pattern.finditer(text))

    candidates = []
    for m in raw_matches:
        prefix = text[max(0, m.start() - 5):m.start()]
        if re.search(r"법\s*$", prefix):
            continue
        num           = int(m.group(1))
        sub           = m.group(2)
        clause_number = f"제{m.group(1)}조" + (f"의{sub}" if sub else "")
        candidates.append((num, clause_number, m.start()))

    header_spans = []
    last_num = 0
    for num, clause_number, start in candidates:
        if num >= last_num and num <= last_num + 5:
            header_spans.append((clause_number, start))
            last_num = num

    def split_into_ho(parent_number: str, unit_text: str) -> list[dict]:
        ho_splits = re.split(HO_SPLIT_PATTERN, unit_text)
        if len(ho_splits) <= 1:
            return [{"clause_number": parent_number, "clause_text": unit_text}]

        head   = ho_splits[0].strip()
        chunks = []
        if head:
            chunks.append({"clause_number": parent_number, "clause_text": head})

        k, last_ho_num = 1, 0
        while k < len(ho_splits) - 1:
            marker       = ho_splits[k].strip()
            ho_num_match = re.match(r"(\d{1,2})\.", marker)
            ho_num       = int(ho_num_match.group(1)) if ho_num_match else (k // 2 + 1)
            ho_body      = ho_splits[k + 1].strip() if k + 1 < len(ho_splits) else ""

            if ho_num == last_ho_num + 1 and ho_body:
                chunks.append({
                    "clause_number": f"{parent_number}제{ho_num}호",
                    "clause_text":   re.sub(r"\s+", " ", f"{marker} {ho_body}").strip(),
                })
                last_ho_num = ho_num
            elif ho_body:
                if chunks:
                    chunks[-1]["clause_text"] += f" {marker} {ho_body}"
                else:
                    chunks.append({"clause_number": parent_number, "clause_text": f"{marker} {ho_body}"})
            k += 2

        return chunks if chunks else [{"clause_number": parent_number, "clause_text": unit_text}]

    clauses = []
    for idx, (clause_number, start) in enumerate(header_spans):
        end       = header_spans[idx + 1][1] if idx + 1 < len(header_spans) else len(text)
        raw_block = text[start:end].strip()

        m          = header_pattern.match(raw_block)
        raw_header = m.group(0) if m else clause_number
        body       = raw_block[m.end():].strip() if m else raw_block

        if not body:
            continue

        hang_splits = re.split(r"([①②③④⑤⑥⑦⑧⑨⑩⑪⑫⑬⑭⑮])", body)

        if len(hang_splits) <= 1:
            clause_text = re.sub(r"\s+", " ", f"{raw_header} {body}").strip()
            clauses.extend(split_into_ho(clause_number, clause_text))
        else:
            j = 1
            while j < len(hang_splits) - 1:
                hang_char   = hang_splits[j]
                hang_body   = hang_splits[j + 1].strip() if j + 1 < len(hang_splits) else ""
                hang_num    = HANG_MAP.get(hang_char, j)
                if hang_body:
                    hang_text = re.sub(r"\s+", " ", f"{raw_header} {hang_char}{hang_body}").strip()
                    clauses.extend(split_into_ho(f"{clause_number}제{hang_num}항", hang_text))
                j += 2

    if not clauses:
        paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
        clauses = [
            {"clause_number": f"단락{i + 1}", "clause_text": para}
            for i, para in enumerate(paragraphs)
        ]

    return clauses


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 공통 검색 / 리랭크 / parent fetch
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _hybrid_search(
    clause_text: str,
    client     : QdrantClient,
    model      : BGEM3FlagModel,
    collection : str,
    fetch_k    : int,
    alpha      : float,
    rrf_k      : int = DEFAULT_RRF_K,
    cache      : SweepCache | None = None,
) -> list[dict]:
    """
    Dense + Sparse 하이브리드 검색 (수동 RRF). alpha=1.0이면 dense만, 0.0이면 sparse만.

    alpha와 rrf_k는 아래 RRF 합산 단계에서만 쓰이므로, Qdrant raw 검색 결과
    (dense_results, sparse_results)는 (collection, clause_text, fetch_k)가
    같으면 alpha/rrf_k와 무관하게 재사용 가능 — cache가 있으면 캐시에서 꺼내온다.
    (fetch_k가 바뀌면 캐시 키 자체가 달라지므로 이 값은 sweep 시 재검색이 발생한다.)
    """
    cache_key = (collection, clause_text, fetch_k)

    if cache is not None and cache_key in cache.raw_search:
        dense_results, sparse_results = cache.raw_search[cache_key]
    else:
        dense_vec, sparse_vec = get_vectors(clause_text, model, cache)
        indices = list(sparse_vec.keys())
        values  = list(sparse_vec.values())

        try:
            dense_results = client.query_points(
                collection_name=collection,
                query=dense_vec,
                using="dense",
                limit=fetch_k,
                with_payload=True,
            ).points

            sparse_results = client.query_points(
                collection_name=collection,
                query=SparseVector(indices=indices, values=values),
                using="sparse",
                limit=fetch_k,
                with_payload=True,
            ).points

        except Exception as e:
            print(f"  ⚠️  sparse 검색 실패, dense만 사용: {e}")
            dense_results = client.query_points(
                collection_name=collection,
                query=dense_vec,
                using="dense",
                limit=fetch_k,
                with_payload=True,
            ).points
            sparse_results = []

        if cache is not None:
            cache.raw_search[cache_key] = (dense_results, sparse_results)

    scores: dict[str, dict] = {}

    for rank, point in enumerate(dense_results, 1):
        cid = point.payload.get("chunk_id", str(point.id))
        scores[cid] = {
            "payload":     point.payload,
            "dense_rank":  rank,
            "sparse_rank": len(dense_results) + 1,
        }

    for rank, point in enumerate(sparse_results, 1):
        cid = point.payload.get("chunk_id", str(point.id))
        if cid in scores:
            scores[cid]["sparse_rank"] = rank
        else:
            scores[cid] = {
                "payload":     point.payload,
                "dense_rank":  len(sparse_results) + 1,
                "sparse_rank": rank,
            }

    results = []
    for cid, info in scores.items():
        rrf_score = (
            alpha         * (1 / (rrf_k + info["dense_rank"]))
            + (1 - alpha) * (1 / (rrf_k + info["sparse_rank"]))
        )
        results.append({
            "chunk_id"    : cid,
            "payload"     : info["payload"],
            "rrf_score"   : rrf_score,
            "is_cross_ref": False,   # 원본 후보 표시 (cross-ref 트리밍에서 구분용)
        })

    results.sort(key=lambda x: x["rrf_score"], reverse=True)
    return results


def _rerank(
    query        : str,
    candidates   : list[dict],
    reranker     : CrossEncoderReranker,
    top_k        : int,
    reranker_name: str = "reranker",
    cache        : SweepCache | None = None,
) -> list[dict]:
    """
    reranker_name은 캐시 키 네임스페이스 구분용 (예: "jo", "ho").
    같은 (reranker_name, query, chunk_id) 조합은 alpha/rrf_k가 달라져도 점수가
    동일하므로, cache가 있으면 미스(cache miss)만 계산하고 나머지는 재사용한다.
    """
    if not candidates:
        return []

    if cache is None:
        texts  = [c["payload"].get("text", c["payload"].get("chunk_text", "")) for c in candidates]
        pairs  = [[query, t] for t in texts]
        scores = reranker.compute_score(pairs, normalize=True)
    else:
        scores: list[float | None] = [None] * len(candidates)
        miss_idx: list[int] = []

        for i, c in enumerate(candidates):
            key = (reranker_name, query, c["chunk_id"])
            if key in cache.rerank_score:
                scores[i] = cache.rerank_score[key]
            else:
                miss_idx.append(i)

        if miss_idx:
            miss_texts = [
                candidates[i]["payload"].get("text", candidates[i]["payload"].get("chunk_text", ""))
                for i in miss_idx
            ]
            miss_pairs  = [[query, t] for t in miss_texts]
            miss_scores = reranker.compute_score(miss_pairs, normalize=True)

            for i, s in zip(miss_idx, miss_scores):
                scores[i] = s
                cache.rerank_score[(reranker_name, query, candidates[i]["chunk_id"])] = s

    ranked = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)
    return [item for _, item in ranked[:top_k]]


def _fetch_parent_texts(
    candidates: list[dict],
    client    : QdrantClient,
    parent_collection: str = COLLECTION_JO,
    cache     : SweepCache | None = None,
) -> list[dict]:
    """
    각 후보 ho chunk_id에서 derive_jo_id()로 조 단위 chunk_id를 역산해,
    그 조 텍스트를 law_kb_jo에서 조회해 payload["text"]를 교체한다.

    payload의 parent_chunk_id 필드는 쓰지 않는다 (derive_jo_id 함수 docstring 참고
    — 그 필드는 JO 컬렉션이 아니라 HO 컬렉션 자기 자신을 가리키고 있어서 이 용도로
    쓰면 매번 조회가 실패한다). chunk_id 자신의 앞쪽 토큰만으로 역산하는 방식은
    leaf 청크든 조 단위 롤업 청크든 상관없이 항상 올바른 jo_id를 준다.
    """
    jo_ids = list({
        derive_jo_id(c["payload"].get("chunk_id", c["chunk_id"]))
        for c in candidates
    })

    if not jo_ids:
        return candidates

    parent_texts: dict[str, str] = {}
    to_fetch: list[str] = []

    for jid in jo_ids:
        cache_key = (parent_collection, jid)
        if cache is not None and cache_key in cache.scroll:
            payload = cache.scroll[cache_key]
            parent_texts[jid] = payload.get("text", payload.get("chunk_text", ""))
        else:
            to_fetch.append(jid)

    try:
        for jid in to_fetch:
            results = client.scroll(
                collection_name=parent_collection,
                scroll_filter=Filter(
                    must=[FieldCondition(key="chunk_id", match=MatchValue(value=jid))]
                ),
                limit=1,
                with_payload=True,
                with_vectors=False,
            )
            points = results[0]
            if points:
                p = points[0].payload
                parent_texts[jid] = p.get("text", p.get("chunk_text", ""))
                if cache is not None:
                    cache.scroll[(parent_collection, jid)] = p
    except Exception as e:
        print(f"  ⚠️  parent fetch 실패: {e}")
        return candidates

    updated = []
    for c in candidates:
        jid = derive_jo_id(c["payload"].get("chunk_id", c["chunk_id"]))
        if jid in parent_texts:
            updated_payload         = dict(c["payload"])
            updated_payload["text"] = parent_texts[jid]
            updated.append({**c, "payload": updated_payload})
        else:
            updated.append(c)

    return updated


def _build_law_refs(
    candidates : list[dict],
    laws_ref   : dict[str, dict],
    top_k      : int,
    with_xref  : bool = False,
) -> list[LawRef]:
    """
    top_k는 여기서 "이미 순위 매겨진 candidates를 자르기만" 한다 — 재검색이나
    재계산이 전혀 없으므로, 같은 candidates에 대해 top_k=1/3/5/10을 전부
    별도 비용 없이 뽑아낼 수 있다 (sweep 스크립트에서 활용).
    """
    law_refs: list[LawRef] = []
    for c in candidates[:top_k]:
        payload  = c["payload"]
        chunk_id = payload.get("chunk_id", "")
        ref_meta = laws_ref.get(chunk_id, {})

        law_refs.append(LawRef(
            chunk_id    = chunk_id,
            article     = ref_meta.get("article",  payload.get("article_id", "")),
            category    = ref_meta.get("category", payload.get("category", "")),
            law_name    = payload.get("law_name",  ""),
            chunk_text  = payload.get("text", payload.get("chunk_text", "")),
            score       = round(float(c.get("rrf_score", 0.0)), 4),
            is_risk_ref = bool(payload.get("is_risk_ref", False)),
            parent_id   = payload.get("parent_chunk_id", "") or "",
            cross_refs  = payload.get("cross_refs", []) if with_xref else [],
        ))

    return law_refs


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# RAG 1: JoRAG — 조 단위 검색
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _search_jo(
    clause_text  : str,
    client       : QdrantClient,
    model        : BGEM3FlagModel,
    laws_ref     : dict[str, dict],
    reranker     : CrossEncoderReranker | None = None,
    use_reranker : bool  = False,
    top_k        : int   = DEFAULT_TOP_K,
    alpha        : float = DEFAULT_ALPHA,
    fetch_k      : int   = DEFAULT_FETCH_K,
    rerank_k     : int   = DEFAULT_RERANK_K,
    rrf_k        : int   = DEFAULT_RRF_K,
    cache        : SweepCache | None = None,
) -> list[LawRef]:
    """
    JoRAG: law_kb_jo에서 조 단위로 직접 검색.
    parent fetch 없음 — 이미 조 단위가 최상위.
    """
    candidates = _hybrid_search(clause_text, client, model, COLLECTION_JO, fetch_k, alpha, rrf_k, cache)

    if use_reranker and reranker and candidates:
        candidates = _rerank(clause_text, candidates, reranker, rerank_k, "jo", cache)

    return _build_law_refs(candidates, laws_ref, top_k, with_xref=False)


def review_contract_jo(
    contract_text: str,
    client       : QdrantClient,
    model        : BGEM3FlagModel,
    laws_ref     : dict[str, dict] | None = None,
    reranker     : CrossEncoderReranker | None = None,
    use_reranker : bool  = False,
    top_k        : int   = DEFAULT_TOP_K,
    alpha        : float = DEFAULT_ALPHA,
    fetch_k      : int   = DEFAULT_FETCH_K,
    rerank_k     : int   = DEFAULT_RERANK_K,
    rrf_k        : int   = DEFAULT_RRF_K,
    cache        : SweepCache | None = None,
) -> list[ClauseResult]:
    """JoRAG 메인 인터페이스."""
    if laws_ref is None:
        laws_ref = load_laws_ref()

    clauses = chunk_contract(contract_text)
    results : list[ClauseResult] = []
    print(f"[JoRAG] 총 {len(clauses)}개 청크 검색 중...")

    for i, clause in enumerate(clauses, 1):
        print(f"  [{i}/{len(clauses)}] {clause['clause_number']} ...", end="\r")

        law_refs   = _search_jo(
            clause["clause_text"], client, model, laws_ref,
            reranker, use_reranker,
            top_k, alpha, fetch_k, rerank_k, rrf_k, cache,
        )
        categories = list(dict.fromkeys(r.category for r in law_refs if r.category))

        results.append(ClauseResult(
            clause_number=clause["clause_number"],
            clause_text  =clause["clause_text"],
            law_refs     =law_refs,
            categories   =categories,
        ))

    print(f"\n[JoRAG] ✅ 완료")
    return results


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# RAG 2: HoRAG — 호/목/세목 단위 검색 + cross_refs 확장 + parent fetch
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _expand_with_cross_refs(
    candidates: list[dict],
    client    : QdrantClient,
    cache     : SweepCache | None = None,
) -> list[dict]:
    """
    각 후보의 cross_refs(같은 law_kb_ho payload 안의 필드)에 있는
    chunk_id를 추가 조회. 이미 후보에 있는 chunk_id는 중복 추가하지 않음.

    여기서는 개수/토큰 제한을 걸지 않고 전부 모은다 — 토큰 예산 트리밍은
    parent fetch 이후(_trim_cross_refs_by_token_budget)에서 한다. 이 시점의
    텍스트는 아직 호/목 단위라 parent fetch 후 조 전체 텍스트로 바뀌면
    길이가 크게 달라지기 때문에, 여기서 트리밍하면 실제 reranker가 보게 될
    길이 기준과 안 맞다.

    추가된 항목에는 "어떤 원본 후보가 이 조항을 인용했는지"의 rrf_score를
    source_score로 같이 기록해둔다 — 트리밍 시 우선순위(원 후보 점수가 높을수록
    먼저 살림)로 쓰기 위함. 여러 원본 후보가 같은 chunk_id를 인용하면
    source_score가 가장 높은 것을 기록한다.

    chunk_id -> payload 조회라 쿼리/alpha와 무관 — 여러 조항이 같은 참조를
    가지면 캐시에서 재사용된다.
    """
    existing_ids = {c["chunk_id"] for c in candidates}
    best_source_score: dict[str, float] = {}

    for c in candidates:
        cross_refs = c["payload"].get("cross_refs", [])
        for ref_id in cross_refs:
            if ref_id in existing_ids:
                continue
            prev = best_source_score.get(ref_id, float("-inf"))
            if c["rrf_score"] > prev:
                best_source_score[ref_id] = c["rrf_score"]

    ref_ids_total = list(best_source_score.keys())
    if not ref_ids_total:
        return candidates

    payload_by_ref: dict[str, dict] = {}
    to_fetch: list[str] = []

    for ref_id in ref_ids_total:
        cache_key = (COLLECTION_HO, ref_id)
        if cache is not None and cache_key in cache.scroll:
            payload_by_ref[ref_id] = cache.scroll[cache_key]
        else:
            to_fetch.append(ref_id)

    try:
        for ref_id in to_fetch:
            results = client.scroll(
                collection_name=COLLECTION_HO,
                scroll_filter=Filter(
                    must=[FieldCondition(key="chunk_id", match=MatchValue(value=ref_id))]
                ),
                limit=1,
                with_payload=True,
                with_vectors=False,
            )
            points = results[0]
            if points:
                p = points[0].payload
                payload_by_ref[ref_id] = p
                if cache is not None:
                    cache.scroll[(COLLECTION_HO, ref_id)] = p
    except Exception as e:
        print(f"  ⚠️  cross_ref fetch 실패: {e}")

    extra_chunks = []
    for ref_id, p in payload_by_ref.items():
        extra_chunks.append({
            "chunk_id"    : ref_id,
            "payload"     : p,
            "rrf_score"   : 0.0,                       # 리랭크에서 점수 재계산됨
            "is_cross_ref": True,                       # 트리밍 대상 표시
            "source_score": best_source_score[ref_id],   # 우선순위 기준
        })

    return candidates + extra_chunks


def _trim_cross_refs_by_token_budget(
    candidates: list[dict],
    reranker  : CrossEncoderReranker,
    max_tokens: int | None,
) -> list[dict]:
    """
    parent fetch 이후(조 전체 텍스트로 교체된 뒤)에 호출한다.

    원본 후보(is_cross_ref=False, hybrid search로 직접 찾은 것)는 무조건
    유지하고, cross-ref로 추가된 후보만 source_score(이 조항을 인용한 원본
    후보의 rrf_score) 내림차순으로 정렬해서, 누적 토큰 수가 max_tokens를
    넘기 전까지만 살린다.

    max_tokens=None이면 트리밍 없이 그대로 반환 (기존 동작과 동일).
    """
    if max_tokens is None:
        return candidates

    originals = [c for c in candidates if not c.get("is_cross_ref")]
    extras    = [c for c in candidates if c.get("is_cross_ref")]

    if not extras:
        return candidates

    extras.sort(key=lambda c: c.get("source_score", 0.0), reverse=True)

    used_tokens = sum(
        reranker.count_tokens(c["payload"].get("text", c["payload"].get("chunk_text", "")))
        for c in originals
    )

    kept_extras = []
    for c in extras:
        text = c["payload"].get("text", c["payload"].get("chunk_text", ""))
        n_tokens = reranker.count_tokens(text)
        if used_tokens + n_tokens > max_tokens:
            continue
        used_tokens += n_tokens
        kept_extras.append(c)

    return originals + kept_extras


def _search_ho(
    clause_text  : str,
    client       : QdrantClient,
    model        : BGEM3FlagModel,
    laws_ref     : dict[str, dict],
    reranker     : CrossEncoderReranker | None = None,
    use_reranker : bool  = False,
    use_cross_refs: bool = True,
    max_cross_ref_tokens: int | None = DEFAULT_MAX_CROSS_REF_TOKENS,
    top_k        : int   = DEFAULT_TOP_K,
    alpha        : float = DEFAULT_ALPHA,
    fetch_k      : int   = DEFAULT_FETCH_K,
    rerank_k     : int   = DEFAULT_RERANK_K,
    rrf_k        : int   = DEFAULT_RRF_K,
    cache        : SweepCache | None = None,
) -> list[LawRef]:
    """
    HoRAG: law_kb_ho에서 호/목/세목 단위 검색
    → (옵션) cross_refs로 관련 조항 확장 (무제한으로 일단 모음)
    → parent_chunk_id로 law_kb_jo에서 조 전체 텍스트 fetch
      (최종 출력 단위는 항상 조 — 하위 단위는 검색 후보로만 쓰고 결과로는 안 남김)
    → (옵션) cross-ref로 추가된 후보만 토큰 예산 기준으로 트리밍
      (원본 후보는 항상 유지, parent fetch 이후 실제 조 텍스트 길이 기준으로 자름)
    → reranker
    """
    candidates = _hybrid_search(clause_text, client, model, COLLECTION_HO, fetch_k, alpha, rrf_k, cache)

    if use_cross_refs:
        candidates = _expand_with_cross_refs(candidates, client, cache)

    # parent fetch: 호/목/세목 → 조 텍스트로 교체 (최종 출력 단위 통일)
    candidates = _fetch_parent_texts(candidates, client, parent_collection=COLLECTION_JO, cache=cache)

    if use_cross_refs and max_cross_ref_tokens is not None:
        if reranker is not None:
            candidates = _trim_cross_refs_by_token_budget(candidates, reranker, max_cross_ref_tokens)
        else:
            print("  ⚠️  max_cross_ref_tokens가 설정됐지만 reranker가 없어 토큰 트리밍을 건너뜁니다.")

    if use_reranker and reranker and candidates:
        candidates = _rerank(clause_text, candidates, reranker, rerank_k, "ho", cache)

    return _build_law_refs(candidates, laws_ref, top_k, with_xref=use_cross_refs)


def review_contract_ho(
    contract_text: str,
    client       : QdrantClient,
    model        : BGEM3FlagModel,
    laws_ref     : dict[str, dict] | None = None,
    reranker     : CrossEncoderReranker | None = None,
    use_reranker : bool  = False,
    use_cross_refs: bool = True,
    max_cross_ref_tokens: int | None = DEFAULT_MAX_CROSS_REF_TOKENS,
    top_k        : int   = DEFAULT_TOP_K,
    alpha        : float = DEFAULT_ALPHA,
    fetch_k      : int   = DEFAULT_FETCH_K,
    rerank_k     : int   = DEFAULT_RERANK_K,
    rrf_k        : int   = DEFAULT_RRF_K,
    cache        : SweepCache | None = None,
) -> list[ClauseResult]:
    """HoRAG 메인 인터페이스 (cross_refs 확장 포함, 별도 xref variant 없음)."""
    if laws_ref is None:
        laws_ref = load_laws_ref()

    clauses = chunk_contract(contract_text)
    results : list[ClauseResult] = []
    print(f"[HoRAG] 총 {len(clauses)}개 청크 검색 중...")

    for i, clause in enumerate(clauses, 1):
        print(f"  [{i}/{len(clauses)}] {clause['clause_number']} ...", end="\r")

        law_refs   = _search_ho(
            clause["clause_text"], client, model, laws_ref,
            reranker, use_reranker, use_cross_refs, max_cross_ref_tokens,
            top_k, alpha, fetch_k, rerank_k, rrf_k, cache,
        )
        categories = list(dict.fromkeys(r.category for r in law_refs if r.category))

        results.append(ClauseResult(
            clause_number=clause["clause_number"],
            clause_text  =clause["clause_text"],
            law_refs     =law_refs,
            categories   =categories,
        ))

    print(f"\n[HoRAG] ✅ 완료")
    return results


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# JSON 변환
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def results_to_json(results: list[ClauseResult]) -> list[dict]:
    return [asdict(r) for r in results]


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 편의: 단일 조항 검색 (sweep 스크립트에서 이 함수들을 직접 호출)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def search_jo(clause_text: str, client: QdrantClient, model: BGEM3FlagModel,
              laws_ref: dict, reranker=None, use_reranker=False,
              top_k=DEFAULT_TOP_K, alpha=DEFAULT_ALPHA, fetch_k=DEFAULT_FETCH_K,
              rerank_k=DEFAULT_RERANK_K, rrf_k=DEFAULT_RRF_K,
              cache: SweepCache | None = None) -> list[LawRef]:
    return _search_jo(clause_text, client, model, laws_ref, reranker,
                       use_reranker, top_k, alpha, fetch_k, rerank_k, rrf_k,
                       cache)


def search_ho(clause_text: str, client: QdrantClient, model: BGEM3FlagModel,
              laws_ref: dict, reranker=None, use_reranker=False, use_cross_refs=True,
              max_cross_ref_tokens=DEFAULT_MAX_CROSS_REF_TOKENS,
              top_k=DEFAULT_TOP_K, alpha=DEFAULT_ALPHA, fetch_k=DEFAULT_FETCH_K,
              rerank_k=DEFAULT_RERANK_K, rrf_k=DEFAULT_RRF_K,
              cache: SweepCache | None = None) -> list[LawRef]:
    return _search_ho(clause_text, client, model, laws_ref, reranker,
                       use_reranker, use_cross_refs, max_cross_ref_tokens,
                       top_k, alpha, fetch_k, rerank_k, rrf_k, cache)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 프로덕션 인터페이스 — 확정된 조합(BEST_JO_CONFIG / BEST_HO_CONFIG)을 그대로
# 적용한 고수준 함수. 호출부에서 alpha/reranker on-off 조합을 매번 기억할
# 필요 없이 이 두 함수만 쓰면 된다.
#
#   review_contract()         : 메인 경로 — 계약서 전체를 조 단위로 검토.
#   get_detailed_citations()  : 보조 경로 — 계약서 조항 1개에 대해 호/목 단위
#                                 세부 근거가 필요할 때만 호출 (사용자가 "세부
#                                 근거 보기" 등을 요청했을 때 온디맨드로 사용).
#
# 주의: reranker가 하나로 통합됐으므로, 재sweep 후 BEST_JO_CONFIG/BEST_HO_CONFIG의
# fetch_k/rerank_k/rrf_k/max_cross_ref_tokens 값을 확정해서 채워 넣을 것.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def review_contract(
    contract_text: str,
    client       : QdrantClient,
    model        : BGEM3FlagModel,
    reranker     : CrossEncoderReranker,
    laws_ref     : dict[str, dict] | None = None,
    top_k        : int = DEFAULT_TOP_K,
    cache        : SweepCache | None = None,
) -> list[ClauseResult]:
    """메인 검토 경로. BEST_JO_CONFIG를 그대로 적용한 JoRAG 호출이다."""
    return review_contract_jo(
        contract_text, client, model, laws_ref,
        reranker=reranker,
        top_k=top_k, cache=cache,
        **BEST_JO_CONFIG,
    )


def get_detailed_citations(
    clause_text  : str,
    client       : QdrantClient,
    model        : BGEM3FlagModel,
    reranker     : CrossEncoderReranker,
    laws_ref     : dict[str, dict] | None = None,
    top_k        : int = DEFAULT_TOP_K,
    cache        : SweepCache | None = None,
) -> list[LawRef]:
    """
    보조 경로. 계약서 조항 1개(review_contract가 반환한 ClauseResult.clause_text)에
    대해 호/목 단위 세부 근거가 필요할 때만 호출한다. BEST_HO_CONFIG를 그대로
    적용한 HoRAG 호출이다.

    review_contract()와 매 요청마다 같이 부르지 말 것 — 세부 근거는 사용자가
    명시적으로 요청했을 때만 온디맨드로 호출하는 게 설계 의도다 (하이브리드
    서비스 구조 — Notion 문서 참고).
    """
    if laws_ref is None:
        laws_ref = load_laws_ref()

    return search_ho(
        clause_text, client, model, laws_ref,
        reranker=reranker,
        top_k=top_k, cache=cache,
        **BEST_HO_CONFIG,
    )


Writing /content/yoonha_law_rag.py


In [5]:
%%writefile /content/yoonha_colab_upsert.py
"""
Workit - Colab용 로컬 Qdrant 셋업 + upsert 스크립트
파일명: rag/yoonha_colab_upsert.py

목적:
  Colab은 로컬 머신의 Qdrant(localhost:6333)에 접근할 수 없다. 대신 이미
  임베딩이 끝난 4개 파일(vectors_*.npz, sparse_weights_*.json)과 payload
  메타데이터(chunks_*.json)를 Colab에 직접 업로드해서, Colab
  안에서 "로컬 파일 기반" Qdrant를 새로 띄우고 그 안에 upsert한다.
  서버 프로세스가 따로 필요 없다 (QdrantClient(path=...) 모드).

  이렇게 하면:
    - ngrok/터널링 없이 Colab 세션 안에서 완결됨
    - 로컬 머신을 계속 켜둘 필요 없음
    - GPU는 reranker/embedding 계산에만 쓰이고, Qdrant 자체는 CPU로 충분

전제:
  - vectors_jo.npz / vectors_ho.npz : {"vectors": (N,1024) float16,
    "chunk_ids": (N,) 문자열 배열}
  - sparse_weights_jo.json / sparse_weights_ho.json : 길이 N인
    리스트, vectors의 chunk_ids와 "같은 순서"로 정렬돼 있다고 이미 확인함
    (jo/ho 둘 다 순서 일치, 중복 0 — 실제 업로드 파일로 검증 완료).
  - chunks_jo.json / chunks_ho.json : chunk_id별 payload
    메타데이터(text, parent_chunk_id, cross_refs, law_name, article_id,
    title, hierarchy, is_ref_article, is_upper_law 등). vectors의 chunk_ids와
    "순서까지" 동일하다고 확인했지만, 이 스크립트는 순서에 의존하지 않고
    chunk_id로 다시 매핑해서 병합한다 (더 안전한 방식).

  주의 — 지금 chunks_*.json에는 category, is_risk_ref 필드가 없다.
  yoonha_law_rag.py의 _build_law_refs가 이 필드들을 payload.get(key, 기본값)
  으로 읽기 때문에 없어도 에러는 안 나고 그냥 빈 문자열/False로 채워진다.
  나중에 laws_ref.json(load_laws_ref)에서 category를 따로 채워주는 구조라
  RAG 파이프라인 동작 자체엔 문제 없음.

사용법 (Colab 셀에서):
    !pip install qdrant-client --quiet
    !python yoonha_colab_upsert.py
  또는 노트북 안에서 build_local_qdrant() 함수를 직접 호출해도 된다.
"""

from __future__ import annotations

import hashlib

import json
from pathlib import Path

import numpy as np
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    SparseVectorParams,
    PointStruct,
    SparseVector,
)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 경로 — Colab에서 업로드한 파일 위치에 맞게 수정
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
DATA_DIR = Path("/content")  # 업로드한 4+2개 파일이 있는 디렉터리

VECTORS_JO      = DATA_DIR / "vectors_jo.npz"
VECTORS_HO      = DATA_DIR / "vectors_ho.npz"
SPARSE_JO       = DATA_DIR / "sparse_weights_jo.json"
SPARSE_HO       = DATA_DIR / "sparse_weights_ho.json"
CHUNKS_JO       = DATA_DIR / "chunks_jo.json"
CHUNKS_HO       = DATA_DIR / "chunks_ho.json"

# Colab 세션 안에 로컬로 만들 Qdrant 저장 경로 (서버 프로세스 없이 파일 기반)
QDRANT_LOCAL_PATH = "/content/qdrant_local"

COLLECTION_JO = "law_kb_jo"
COLLECTION_HO = "law_kb_ho"

DENSE_DIM = 1024  # BGE-M3 dense 차원 (npz vectors.shape[1]로 실제 검증도 함께 함)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 로드 / 병합
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _load_merged(vectors_path: Path, sparse_path: Path, chunks_path: Path) -> list[dict]:
    """
    vectors(.npz) + sparse_weights(.json) + chunks(.json, payload)를
    chunk_id 기준으로 병합해서 [{chunk_id, dense_vec, sparse_vec, payload}, ...] 반환.
    """
    npz = np.load(vectors_path, allow_pickle=True)
    dense_vectors = npz["vectors"]          # (N, 1024) float16
    chunk_ids     = npz["chunk_ids"].tolist()

    assert dense_vectors.shape[1] == DENSE_DIM, (
        f"예상한 dense 차원({DENSE_DIM})과 실제({dense_vectors.shape[1]})가 다릅니다 — "
        f"임베딩 모델이 바뀐 건 아닌지 확인하세요."
    )

    with open(sparse_path, encoding="utf-8") as f:
        sparse_weights = json.load(f)  # chunk_ids와 같은 순서의 리스트

    assert len(sparse_weights) == len(chunk_ids), (
        f"sparse_weights 개수({len(sparse_weights)})와 chunk_ids 개수({len(chunk_ids)})가 다릅니다."
    )

    with open(chunks_path, encoding="utf-8") as f:
        chunks_list = json.load(f)

    payload_by_id = {c["chunk_id"]: c for c in chunks_list}

    missing = [cid for cid in chunk_ids if cid not in payload_by_id]
    if missing:
        print(f"  ⚠️  payload를 못 찾은 chunk_id {len(missing)}개 (예: {missing[:5]}) "
              f"— 해당 chunk는 text 없이 upsert됩니다.")

    merged = []
    for i, cid in enumerate(chunk_ids):
        payload = dict(payload_by_id.get(cid, {}))
        payload["chunk_id"] = cid  # payload 안에도 chunk_id를 넣어야 검색 결과에서 식별 가능
        merged.append({
            "chunk_id":   cid,
            "dense_vec":  dense_vectors[i].astype(np.float32).tolist(),
            "sparse_vec": sparse_weights[i],  # {"token_id": weight, ...}
            "payload":    payload,
        })

    return merged


def _to_qdrant_points(merged: list[dict]) -> list[PointStruct]:
    points = []
    for item in merged:
        sparse = item["sparse_vec"]
        indices = [int(k) for k in sparse.keys()]
        values  = [float(v) for v in sparse.values()]

        points.append(PointStruct(
            id=int(hashlib.sha256(item["chunk_id"].encode("utf-8")).hexdigest(), 16) % (2**63),  # 결정론적 해시(hashlib) — 내장 hash()는 PYTHONHASHSEED로 실행마다 값이 달라질 수 있어 교체함
            vector={
                "dense":  item["dense_vec"],
                "sparse": SparseVector(indices=indices, values=values),
            },
            payload=item["payload"],
        ))
    return points


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 컬렉션 생성 + upsert
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _create_collection(client: QdrantClient, name: str) -> None:
    if client.collection_exists(name):
        print(f"  ℹ️  {name} 이미 존재 — 삭제 후 재생성")
        client.delete_collection(name)

    client.create_collection(
        collection_name=name,
        vectors_config={
            "dense": VectorParams(size=DENSE_DIM, distance=Distance.COSINE),
        },
        sparse_vectors_config={
            "sparse": SparseVectorParams(),
        },
    )


def _upsert_batched(client: QdrantClient, name: str, points: list[PointStruct], batch_size: int = 256) -> None:
    total = len(points)
    for i in range(0, total, batch_size):
        batch = points[i : i + batch_size]
        client.upsert(collection_name=name, points=batch)
        print(f"  [{name}] {min(i + batch_size, total)}/{total} upsert 완료", end="\r")
    print()


def build_local_qdrant(recreate: bool = True) -> QdrantClient:
    """
    Colab 세션 안에서 로컬(파일 기반) Qdrant를 만들고, 업로드된 4+2개 파일로
    law_kb_jo / law_kb_ho 컬렉션을 채운다.
    반환된 client를 그대로 yoonha_rag_eval*.py의 QdrantClient 자리에 넣으면 된다.
    """
    client = QdrantClient(path=QDRANT_LOCAL_PATH)

    print("📦 jo 데이터 로드/병합 중...")
    merged_jo = _load_merged(VECTORS_JO, SPARSE_JO, CHUNKS_JO)
    print(f"  -> {len(merged_jo)}개 chunk 병합 완료")

    print("📦 ho 데이터 로드/병합 중...")
    merged_ho = _load_merged(VECTORS_HO, SPARSE_HO, CHUNKS_HO)
    print(f"  -> {len(merged_ho)}개 chunk 병합 완료")

    print(f"🗂️  {COLLECTION_JO} 컬렉션 생성...")
    _create_collection(client, COLLECTION_JO)
    _upsert_batched(client, COLLECTION_JO, _to_qdrant_points(merged_jo))

    print(f"🗂️  {COLLECTION_HO} 컬렉션 생성...")
    _create_collection(client, COLLECTION_HO)
    _upsert_batched(client, COLLECTION_HO, _to_qdrant_points(merged_ho))

    jo_count = client.count(COLLECTION_JO).count
    ho_count = client.count(COLLECTION_HO).count
    print(f"\n✅ 완료 — {COLLECTION_JO}: {jo_count}개, {COLLECTION_HO}: {ho_count}개")

    return client


if __name__ == "__main__":
    build_local_qdrant()



Writing /content/yoonha_colab_upsert.py


## 3. 업로드 파일 존재 확인

In [6]:
from pathlib import Path

REQUIRED_FILES = [
    "/content/vectors_jo.npz",
    "/content/vectors_ho.npz",
    "/content/sparse_weights_jo.json",
    "/content/sparse_weights_ho.json",
    "/content/chunks_jo.json",
    "/content/chunks_ho.json",
    "/content/eval_set.json",  # 50개/100개 등 파일명이 다르면 여기만 바꾸세요
]

missing = [f for f in REQUIRED_FILES if not Path(f).exists()]
if missing:
    print("⚠️  아직 안 올라온 파일:")
    for f in missing:
        print("  -", f)
    print("\n왼쪽 파일 탐색기(📁)에서 위 파일들을 /content/에 업로드한 뒤 이 셀을 다시 실행하세요.")
else:
    print("✅ 필요한 파일이 모두 있습니다.")

✅ 필요한 파일이 모두 있습니다.


## 4. 로컬 Qdrant 생성 + upsert

`/content/qdrant_local`에 파일 기반 Qdrant를 만들고,
`law_kb_jo`, `law_kb_ho` 컬렉션을 채웁니다.
서버 프로세스가 따로 필요 없습니다.

In [7]:
import sys
sys.path.insert(0, "/content")

from yoonha_colab_upsert import build_local_qdrant

client = build_local_qdrant()

# 로컬(파일 기반) Qdrant는 경로 하나에 클라이언트가 동시에 하나만 붙을 수 있음.
# 아래 sweep 셀은 !python으로 "별도 프로세스"에서 같은 경로를 다시 여니까,
# 여기서 만든 client를 계속 열어두면 그쪽에서 파일 락 충돌(AlreadyLocked)이 남.
# upsert가 끝났으면 바로 닫아서 락을 풀어준다.
client.close()
print("✅ upsert 완료, client 닫음 (sweep 프로세스가 같은 경로에 새로 붙을 수 있게)")

📦 jo 데이터 로드/병합 중...
  -> 1186개 chunk 병합 완료
📦 ho 데이터 로드/병합 중...
  -> 7640개 chunk 병합 완료
🗂️  law_kb_jo 컬렉션 생성...
  [law_kb_jo] 1186/1186 upsert 완료
🗂️  law_kb_ho 컬렉션 생성...
  [law_kb_ho] 7640/7640 upsert 완료

✅ 완료 — law_kb_jo: 1186개, law_kb_ho: 7640개
✅ upsert 완료, client 닫음 (sweep 프로세스가 같은 경로에 새로 붙을 수 있게)


## 5. 단계적(Staged) sweep 실행 (reranker ON/OFF 둘 다)

`QDRANT_LOCAL_PATH`, `RERANKER_DEVICE` 환경변수만 잡아주면
`yoonha_rag_eval_staged.py` 안의 client/reranker 생성 로직이 자동으로
Colab용(로컬 파일 기반 Qdrant + GPU)으로 분기합니다.

**reranker ON (프로덕션 후보):**
- 0단계: reranker OFF baseline (참고용, 기본 alpha/rrf_k 고정)
- 1단계: alpha × rrf_k (20개)
- 2단계: fetch_k × rerank_k (rerank_k ≤ fetch_k만, 최대 10개)
- 3단계 (HoRAG만): use_cross_refs × max_cross_ref_tokens (5개)

**reranker OFF (신규 — "리랭커가 얼마나 기여하는지" 비교용):**
- 1단계_off: alpha × rrf_k를 reranker OFF 기준으로 처음부터 재sweep
  (reranker가 꺼지면 최적 alpha가 크게 달라질 수 있어서 ON에서 찾은 값을
  재사용하지 않습니다 — v3 문서에서 확인된 패턴)
- 2단계_off: fetch_k만 (rerank_k 축은 리랭커가 없으니 의미가 없어 제외)
- 3단계_off (HoRAG만): use_cross_refs × max_cross_ref_tokens (ON과 동일 그리드)

이 셀 하나로 jo/ho 각각 on/off, 총 4개 확정 조합이 `eval_final_best.csv`에
쌓입니다 (cascaded on/off 2개는 8번 섹션에서 추가되어 최종 6행이 됩니다).


In [8]:
%%writefile /content/yoonha_rag_eval_staged.py
"""
Workit - RAG 파라미터 단계적(staged) sweep, reranker 단일화 버전
파일명: rag/yoonha_rag_eval_staged.py

완전 교차곱(alpha × rrf_k × fetch_k × rerank_k × ...)을 한 번에 다 돌리면
JoRAG 380개, HoRAG 760개까지 불어나서(2280개까지도 감), 대신 아래처럼
단계적으로 좁혀서 돈다.

reranker ON (기존, 프로덕션 후보):
  0단계 (baseline)  : reranker 아예 끈 raw hybrid search 성능 확인 (참고용,
                       alpha/rrf_k는 기본값 고정 — off 전용 sweep과는 다름)
  1단계 (alpha×rrf_k): fetch_k/rerank_k는 기본값에 고정하고,
                       alpha(5) × rrf_k(4) = 20개만 sweep.
  2단계 (fetch_k×rerank_k): 1단계 최적 alpha/rrf_k를 고정하고
                       fetch_k(4) × rerank_k(4, rerank_k<=fetch_k 필터) sweep.
  3단계 (HoRAG 전용, cross-ref): use_cross_refs on/off + max_cross_ref_tokens(4개).

reranker OFF (신규 — "리랭커가 실제로 얼마나 기여하는지" 비교용):
  리랭커를 아예 끈 상태에서도 "가능한 축은 전부" 다시 sweep한다. 리랭커가
  꺼지면 최종 순위가 순수 RRF 점수로만 결정되므로, rerank_k 축은 의미가
  없어져서 제외하고 alpha×rrf_k, fetch_k만 sweep한다 (HoRAG는 cross-ref도
  동일하게 sweep). v3 문서에서 확인됐듯 reranker가 꺼지면 최적 alpha가
  크게 달라질 수 있어서(alpha=1.0 dense-only가 유리해지는 경향), off 전용
  으로 alpha/rrf_k를 처음부터 다시 찾는다 — reranker ON에서 찾은 값을
  그대로 재사용하지 않는다.
  1단계_off (alpha×rrf_k) : fetch_k 기본값 고정, use_reranker=False로 sweep.
  2단계_off (fetch_k)     : 1단계_off 최적 alpha/rrf_k 고정, fetch_k만 sweep
                            (rerank_k 축 없음 — 리랭커가 없으니 "몇 개를
                            리랭크할지" 자체가 무의미).
  3단계_off (HoRAG 전용, cross-ref): reranker ON과 동일한 그리드로 sweep.
                            (max_cross_ref_tokens 트리밍은 리랭커 "점수"가
                            아니라 리랭커의 tokenizer로 토큰 수만 세는
                            것이라 use_reranker=False에서도 그대로 동작한다.)

이 구조로 돌면 (on + off 합산):
  JoRAG : on 36개 + off 1(baseline) + 20(1단계_off) + 4(2단계_off) ≈ 61개
  HoRAG : 위 + 3단계(on 5개 + off 5개) ≈ 71개

reranker는 bge-reranker-v2-m3 하나로 고정 (yoonha_law_rag.py 개정판 기준).

평가 기준 (2026-07 개정): LLM에는 결국 "조" 텍스트가 최종적으로 전달되므로
(HoRAG도 parent fetch로 조 텍스트를 붙여서 반환), jo/ho 두 variant 모두
GT를 relevant_docs_jo(조 단위)로 통일했다. ho variant는 raw 후보를 넉넉히
뽑은 뒤 derive_jo_id()로 조 단위로 접어(중복 제거) top-k를 계산한다.

  Jo는 LLM에 top-2~3개만 넘기는 용도라 최적 조합 선정 기준을
  Recall@3 (동률 시 MRR)으로 잡는다. Ho는 top-10까지 넘기므로
  기존처럼 MRR 기준을 유지한다. on/off 모두 동일한 기준을 쓴다 —
  "리랭커를 껐을 때 최선"과 "켰을 때 최선"을 같은 기준으로 비교해야
  공정한 비교이기 때문이다.

  Recall@k는 gt_docs 중 "하나라도" top-k에 걸리면 hit으로 치는 지표라
  멀티독(정답이 여러 개인 쿼리) 케이스에서 "top-k 안에 정답이 다 담기는지"는
  못 잡아낸다. 이를 보완하기 위해 FullCoverage@k(정답 전체가 top-k에
  subset으로 포함되는 비율)를 함께 계산한다.

출력:
  - eval_stage0_baseline.csv               (reranker ON 파이프라인의 참고용 baseline)
  - eval_stage1_alpha_rrf_{jo,ho}.csv       (reranker ON)
  - eval_stage2_fetchk_rerankk_{jo,ho}.csv  (reranker ON)
  - eval_stage3_crossref_ho.csv             (reranker ON)
  - eval_stage1_off_alpha_rrf_{jo,ho}.csv   (reranker OFF, 신규)
  - eval_stage2_off_fetchk_{jo,ho}.csv      (reranker OFF, 신규)
  - eval_stage3_off_crossref_ho.csv         (reranker OFF, 신규)
  - eval_final_best.csv  (variant × use_reranker 조합별 최종 확정 조합 —
                           jo/ho 각각 on/off 2행씩 총 4행. cascaded on/off는
                           yoonha_rag_eval_cascaded_staged.py에서 2행 추가돼서
                           최종적으로 6행이 된다.)

사용법:
    pip install qdrant-client FlagEmbedding transformers torch pandas
    python rag/yoonha_rag_eval_staged.py
"""

from __future__ import annotations

import itertools
import json
import os
import time
from pathlib import Path

import pandas as pd
from qdrant_client import QdrantClient

from yoonha_law_rag import (
    load_embed_model,
    load_laws_ref,
    load_reranker,
    search_jo,
    search_ho,
    derive_jo_id,
    SweepCache,
    QDRANT_HOST,
    QDRANT_PORT,
    DEFAULT_FETCH_K,
    DEFAULT_RERANK_K,
    DEFAULT_RRF_K,
    DEFAULT_ALPHA,
    DEFAULT_MAX_CROSS_REF_TOKENS,
)

# Colab 등 로컬 서버(localhost:6333)에 못 붙는 환경에서는
# QDRANT_LOCAL_PATH 환경변수를 잡아주면 파일 기반 로컬 Qdrant를 쓴다.
# GPU 리랭커를 쓰려면 RERANKER_DEVICE=cuda로 잡아준다 (기본값 cpu).
QDRANT_LOCAL_PATH_ENV = os.environ.get("QDRANT_LOCAL_PATH")
RERANKER_DEVICE = os.environ.get("RERANKER_DEVICE", "cpu")


def get_qdrant_client() -> QdrantClient:
    if QDRANT_LOCAL_PATH_ENV:
        print(f"🗄️  로컬(파일 기반) Qdrant 사용: {QDRANT_LOCAL_PATH_ENV}")
        return QdrantClient(path=QDRANT_LOCAL_PATH_ENV)
    print(f"🗄️  서버 Qdrant 사용: {QDRANT_HOST}:{QDRANT_PORT}")
    return QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)


GOLD_STANDARD_PATH = Path("/content/eval_set.json")  # 50개든 100개든 경로만 이거면 됨
OUT_DIR = Path("/content")
CACHE_PATH = OUT_DIR / "sweep_cache_staged.pkl"  # 세션 끊겨도 이어받을 수 있게 디스크에 저장

# LLM에는 항상 "조" 텍스트가 최종적으로 전달된다 (HoRAG도 parent fetch로
# 조 텍스트를 붙여서 내보냄 — chunk_id 자체는 ho-level이지만 실제로 LLM이
# 보는 건 조 전체 텍스트). 그래서 jo/ho 두 variant 모두 "정답 조를 가져왔는가"
# 기준으로 평가한다 — GT는 항상 relevant_docs_jo 하나로 통일.
GT_FIELD_BY_VARIANT = {
    "jo": "relevant_docs_jo",
    "ho": "relevant_docs_jo",
}

# 최종 recall을 볼 top_k 목록 — 한 번 검색한 결과에서 전부 잘라 계산 (재검색 없음)
# 2를 추가한 이유: Jo가 top-2~3만 LLM에 넘기는 용도라 top-2 단독 성능도 봐야 함
RECALL_AT_KS = [1, 2, 3, 5, 10]
TOP_K_EVAL   = max(RECALL_AT_KS)

# ho는 chunk_id가 호/목/세목 단위라 여러 개가 같은 조로 접힐 수 있다
# (예: top-10 raw ho 후보가 사실은 조 3개짜리일 수 있음). "조 기준 top-k"를
# 공정하게 재려면 raw 후보를 TOP_K_EVAL보다 넉넉히 뽑아서 조 단위로 접은
# 뒤에 잘라야 한다. rerank_k/fetch_k sweep 그리드의 최댓값(30)과 맞춰
# 여유를 준다 — 추가 검색/재계산 비용은 없다(이미 만들어진 candidates
# 리스트를 얼마나 자르느냐만 다름).
RAW_TOP_K_FOR_JO_DEDUP = 30

# Jo/Ho 각각 최적 조합을 고를 때 기준으로 삼을 컬럼. on/off 모두 동일 기준 사용.
JO_SORT_COL = "Recall@3"
HO_SORT_COL = "MRR"

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 각 단계 sweep 그리드
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ALPHA_GRID    = [0.3, 0.5, 0.6, 0.7, 1.0]   # 1.0 = dense only
RRF_K_GRID    = [20, 40, 60, 100]           # 60 = 기존 하드코딩값
FETCH_K_GRID  = [20, 30, 50, 80]
RERANK_K_GRID = [5, 10, 20, 30]

USE_CROSS_REFS_GRID = [True, False]
MAX_CROSS_REF_TOKENS_GRID = [500, 1000, 2000, None]  # None = 트리밍 없음(무제한)


def load_gold_standard(path: Path = GOLD_STANDARD_PATH) -> list[dict]:
    with open(path, encoding="utf-8") as f:
        return json.load(f)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 공통 평가 함수 — variant(jo/ho)에 따라 search_jo/search_ho만 다르게 호출
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def evaluate_params(
    variant      : str,
    client       : QdrantClient,
    model,
    laws_ref     : dict,
    reranker,
    gold_standard: list[dict],
    cache        : SweepCache,
    params       : dict,
) -> dict:
    """
    params는 search_jo/search_ho에 그대로 넘길 키워드 인자 dict
    (alpha, rrf_k, fetch_k, rerank_k, use_reranker, [use_cross_refs, max_cross_ref_tokens]).

    Recall@k       : gt_docs 중 "하나라도" top-k 안에 있으면 hit
    FullCoverage@k : gt_docs "전부"가 top-k 안에 포함되면 hit
                     (멀티독 쿼리에서 top-k를 작게 자를 때 정답 누락 여부를 잡아냄)

    jo/ho 둘 다 GT는 relevant_docs_jo(조 단위)다. ho variant는 chunk_id 자체는
    호/목/세목 단위라, raw 후보를 RAW_TOP_K_FOR_JO_DEDUP개까지 넉넉히 뽑은 뒤
    derive_jo_id()로 조 id로 접고 순서를 유지한 채 중복 제거해서 "조 단위
    top-k"를 만든다 (같은 조에서 나온 여러 호가 top-k 슬롯을 낭비하지 않게).
    jo variant는 chunk_id가 이미 조 단위라 이 과정이 사실상 no-op이다.
    """
    recall_hits = {k: 0 for k in RECALL_AT_KS}
    full_coverage_hits = {k: 0 for k in RECALL_AT_KS}
    mrr = 0.0
    n = len(gold_standard)
    t0 = time.time()
    gt_field = GT_FIELD_BY_VARIANT[variant]

    search_fn = search_jo if variant == "jo" else search_ho

    for item in gold_standard:
        gt_docs = set(item[gt_field])

        law_refs = search_fn(
            clause_text=item["query"],
            client=client,
            model=model,
            laws_ref=laws_ref,
            reranker=reranker,
            top_k=RAW_TOP_K_FOR_JO_DEDUP,
            cache=cache,
            **params,
        )
        raw_ranked_ids = [r.chunk_id for r in law_refs]

        # 조 id로 접고, 순서를 유지한 채 중복만 제거
        seen: set[str] = set()
        ranked_ids: list[str] = []
        for cid in raw_ranked_ids:
            jid = derive_jo_id(cid)
            if jid not in seen:
                seen.add(jid)
                ranked_ids.append(jid)

        for k in RECALL_AT_KS:
            top_k_set = set(ranked_ids[:k])
            if gt_docs & top_k_set:
                recall_hits[k] += 1
            if gt_docs and gt_docs.issubset(top_k_set):
                full_coverage_hits[k] += 1
        for rank, d in enumerate(ranked_ids, 1):
            if d in gt_docs:
                mrr += 1 / rank
                break

    elapsed = time.time() - t0

    result = {"variant": variant, **params}
    for k in RECALL_AT_KS:
        result[f"Recall@{k}"] = round(recall_hits[k] / n, 4)
        result[f"FullCoverage@{k}"] = round(full_coverage_hits[k] / n, 4)
    result["MRR"] = round(mrr / n, 4)
    result["avg_sec_per_query"] = round(elapsed / n, 3)
    return result


def run_grid(
    variant, client, model, laws_ref, reranker, gold_standard, cache,
    fixed_params: dict, grid_params: dict, stage_name: str,
) -> pd.DataFrame:
    """
    fixed_params: 이번 단계에서 고정할 값들
    grid_params : {key: [values...]} 형태, itertools.product로 전부 조합
    """
    keys = list(grid_params.keys())
    value_lists = [grid_params[k] for k in keys]
    combos = list(itertools.product(*value_lists))

    print(f"\n━━━ [{stage_name}] {variant} — {len(combos)}개 조합 ━━━")

    rows = []
    for i, combo_values in enumerate(combos, 1):
        params = {**fixed_params, **dict(zip(keys, combo_values))}
        print(f"  [{i}/{len(combos)}] {params} ...", end="")
        result = evaluate_params(variant, client, model, laws_ref, reranker, gold_standard, cache, params)
        rows.append(result)
        if variant == "jo":
            print(f" -> MRR={result['MRR']} Recall@2={result['Recall@2']} Recall@3={result['Recall@3']} "
                  f"FullCoverage@3={result['FullCoverage@3']} ({result['avg_sec_per_query']}초/쿼리)")
        else:
            print(f" -> MRR={result['MRR']} Recall@10={result['Recall@10']} ({result['avg_sec_per_query']}초/쿼리)")

    df = pd.DataFrame(rows).sort_values("MRR", ascending=False)
    return df


def get_best_row(df: pd.DataFrame, sort_col: str = "MRR") -> dict:
    return df.sort_values(sort_col, ascending=False).iloc[0].to_dict()


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# JoRAG 단계적 sweep — reranker ON (기존)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def sweep_jo(client, model, laws_ref, reranker, gold_standard, cache) -> dict:
    baseline_params = dict(
        alpha=DEFAULT_ALPHA, rrf_k=DEFAULT_RRF_K,
        fetch_k=DEFAULT_FETCH_K, rerank_k=DEFAULT_RERANK_K,
        use_reranker=False,
    )
    baseline = evaluate_params("jo", client, model, laws_ref, reranker, gold_standard, cache, baseline_params)
    print(f"\n[jo baseline, reranker OFF, 기본 alpha/rrf_k] MRR={baseline['MRR']} Recall@3={baseline['Recall@3']} "
          f"FullCoverage@3={baseline['FullCoverage@3']}")

    df_stage1 = run_grid(
        "jo", client, model, laws_ref, reranker, gold_standard, cache,
        fixed_params=dict(fetch_k=DEFAULT_FETCH_K, rerank_k=DEFAULT_RERANK_K, use_reranker=True),
        grid_params=dict(alpha=ALPHA_GRID, rrf_k=RRF_K_GRID),
        stage_name="1단계 alpha×rrf_k (reranker ON)",
    )
    df_stage1.to_csv(OUT_DIR / "eval_stage1_alpha_rrf_jo.csv", index=False)
    best1 = get_best_row(df_stage1, sort_col=JO_SORT_COL)
    print(f"[jo 1단계 최적 ({JO_SORT_COL} 기준)] alpha={best1['alpha']}, rrf_k={best1['rrf_k']} "
          f"-> {JO_SORT_COL}={best1[JO_SORT_COL]} MRR={best1['MRR']}")

    combos_fk_rk = [
        (fk, rk) for fk, rk in itertools.product(FETCH_K_GRID, RERANK_K_GRID) if rk <= fk
    ]
    df_stage2_rows = []
    print(f"\n━━━ [2단계 fetch_k×rerank_k (reranker ON)] jo — {len(combos_fk_rk)}개 조합 ━━━")
    for i, (fk, rk) in enumerate(combos_fk_rk, 1):
        params = dict(
            alpha=best1["alpha"], rrf_k=best1["rrf_k"],
            fetch_k=fk, rerank_k=rk, use_reranker=True,
        )
        print(f"  [{i}/{len(combos_fk_rk)}] fetch_k={fk} rerank_k={rk} ...", end="")
        result = evaluate_params("jo", client, model, laws_ref, reranker, gold_standard, cache, params)
        df_stage2_rows.append(result)
        print(f" -> MRR={result['MRR']} Recall@2={result['Recall@2']} Recall@3={result['Recall@3']} "
              f"FullCoverage@3={result['FullCoverage@3']} ({result['avg_sec_per_query']}초/쿼리)")

    df_stage2 = pd.DataFrame(df_stage2_rows).sort_values("MRR", ascending=False)
    df_stage2.to_csv(OUT_DIR / "eval_stage2_fetchk_rerankk_jo.csv", index=False)
    best2 = get_best_row(df_stage2, sort_col=JO_SORT_COL)
    print(f"[jo 2단계 최적 ({JO_SORT_COL} 기준)] fetch_k={best2['fetch_k']}, rerank_k={best2['rerank_k']} "
          f"-> {JO_SORT_COL}={best2[JO_SORT_COL]} MRR={best2['MRR']}")

    final = dict(
        variant="jo", use_reranker=True,
        alpha=best1["alpha"], rrf_k=best1["rrf_k"],
        fetch_k=best2["fetch_k"], rerank_k=best2["rerank_k"],
        MRR=best2["MRR"],
        **{f"Recall@{k}": best2[f"Recall@{k}"] for k in RECALL_AT_KS},
        **{f"FullCoverage@{k}": best2[f"FullCoverage@{k}"] for k in RECALL_AT_KS},
    )
    return final


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# JoRAG 단계적 sweep — reranker OFF (신규, on과 동일한 기준으로 비교용)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def sweep_jo_off(client, model, laws_ref, reranker, gold_standard, cache) -> dict:
    """
    reranker를 항상 끈 상태로 alpha×rrf_k, fetch_k를 처음부터 다시 찾는다.
    rerank_k 축은 없다 (리랭커가 없으니 "몇 개를 리랭크할지" 자체가 무의미).
    reranker ON에서 찾은 alpha/rrf_k/fetch_k를 그대로 재사용하지 않는 이유는,
    reranker가 꺼지면 최종 순위가 순수 RRF 점수로 결정돼서 최적 alpha가
    크게 달라질 수 있기 때문이다 (v3 문서 참고 — alpha=1.0 dense-only가
    reranker OFF에서 유리해지는 경향이 확인된 바 있음).
    """
    df_stage1_off = run_grid(
        "jo", client, model, laws_ref, reranker, gold_standard, cache,
        fixed_params=dict(fetch_k=DEFAULT_FETCH_K, rerank_k=DEFAULT_RERANK_K, use_reranker=False),
        grid_params=dict(alpha=ALPHA_GRID, rrf_k=RRF_K_GRID),
        stage_name="1단계_off alpha×rrf_k (reranker OFF)",
    )
    df_stage1_off.to_csv(OUT_DIR / "eval_stage1_off_alpha_rrf_jo.csv", index=False)
    best1 = get_best_row(df_stage1_off, sort_col=JO_SORT_COL)
    print(f"[jo 1단계_off 최적 ({JO_SORT_COL} 기준)] alpha={best1['alpha']}, rrf_k={best1['rrf_k']} "
          f"-> {JO_SORT_COL}={best1[JO_SORT_COL]} MRR={best1['MRR']}")

    df_stage2_rows = []
    print(f"\n━━━ [2단계_off fetch_k (reranker OFF)] jo — {len(FETCH_K_GRID)}개 조합 ━━━")
    for i, fk in enumerate(FETCH_K_GRID, 1):
        params = dict(
            alpha=best1["alpha"], rrf_k=best1["rrf_k"],
            fetch_k=fk, rerank_k=DEFAULT_RERANK_K, use_reranker=False,
        )
        print(f"  [{i}/{len(FETCH_K_GRID)}] fetch_k={fk} ...", end="")
        result = evaluate_params("jo", client, model, laws_ref, reranker, gold_standard, cache, params)
        df_stage2_rows.append(result)
        print(f" -> MRR={result['MRR']} Recall@2={result['Recall@2']} Recall@3={result['Recall@3']} "
              f"FullCoverage@3={result['FullCoverage@3']} ({result['avg_sec_per_query']}초/쿼리)")

    df_stage2_off = pd.DataFrame(df_stage2_rows).sort_values("MRR", ascending=False)
    df_stage2_off.to_csv(OUT_DIR / "eval_stage2_off_fetchk_jo.csv", index=False)
    best2 = get_best_row(df_stage2_off, sort_col=JO_SORT_COL)
    print(f"[jo 2단계_off 최적 ({JO_SORT_COL} 기준)] fetch_k={best2['fetch_k']} "
          f"-> {JO_SORT_COL}={best2[JO_SORT_COL]} MRR={best2['MRR']}")

    final = dict(
        variant="jo", use_reranker=False,
        alpha=best1["alpha"], rrf_k=best1["rrf_k"],
        fetch_k=best2["fetch_k"], rerank_k=DEFAULT_RERANK_K,
        MRR=best2["MRR"],
        **{f"Recall@{k}": best2[f"Recall@{k}"] for k in RECALL_AT_KS},
        **{f"FullCoverage@{k}": best2[f"FullCoverage@{k}"] for k in RECALL_AT_KS},
    )
    return final


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# HoRAG 단계적 sweep — reranker ON (기존, 1·2단계는 jo와 동일 구조 + 3단계 cross-ref)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def sweep_ho(client, model, laws_ref, reranker, gold_standard, cache) -> dict:
    baseline_params = dict(
        alpha=DEFAULT_ALPHA, rrf_k=DEFAULT_RRF_K,
        fetch_k=DEFAULT_FETCH_K, rerank_k=DEFAULT_RERANK_K,
        use_reranker=False, use_cross_refs=True,
        max_cross_ref_tokens=DEFAULT_MAX_CROSS_REF_TOKENS,
    )
    baseline = evaluate_params("ho", client, model, laws_ref, reranker, gold_standard, cache, baseline_params)
    print(f"\n[ho baseline, reranker OFF, 기본값] MRR={baseline['MRR']} Recall@10={baseline['Recall@10']}")

    df_stage1 = run_grid(
        "ho", client, model, laws_ref, reranker, gold_standard, cache,
        fixed_params=dict(
            fetch_k=DEFAULT_FETCH_K, rerank_k=DEFAULT_RERANK_K, use_reranker=True,
            use_cross_refs=True, max_cross_ref_tokens=DEFAULT_MAX_CROSS_REF_TOKENS,
        ),
        grid_params=dict(alpha=ALPHA_GRID, rrf_k=RRF_K_GRID),
        stage_name="1단계 alpha×rrf_k (reranker ON)",
    )
    df_stage1.to_csv(OUT_DIR / "eval_stage1_alpha_rrf_ho.csv", index=False)
    best1 = get_best_row(df_stage1, sort_col=HO_SORT_COL)
    print(f"[ho 1단계 최적] alpha={best1['alpha']}, rrf_k={best1['rrf_k']} -> MRR={best1['MRR']}")

    combos_fk_rk = [
        (fk, rk) for fk, rk in itertools.product(FETCH_K_GRID, RERANK_K_GRID) if rk <= fk
    ]
    df_stage2_rows = []
    print(f"\n━━━ [2단계 fetch_k×rerank_k (reranker ON)] ho — {len(combos_fk_rk)}개 조합 ━━━")
    for i, (fk, rk) in enumerate(combos_fk_rk, 1):
        params = dict(
            alpha=best1["alpha"], rrf_k=best1["rrf_k"],
            fetch_k=fk, rerank_k=rk, use_reranker=True,
            use_cross_refs=True, max_cross_ref_tokens=DEFAULT_MAX_CROSS_REF_TOKENS,
        )
        print(f"  [{i}/{len(combos_fk_rk)}] fetch_k={fk} rerank_k={rk} ...", end="")
        result = evaluate_params("ho", client, model, laws_ref, reranker, gold_standard, cache, params)
        df_stage2_rows.append(result)
        print(f" -> MRR={result['MRR']} Recall@10={result['Recall@10']} ({result['avg_sec_per_query']}초/쿼리)")

    df_stage2 = pd.DataFrame(df_stage2_rows).sort_values("MRR", ascending=False)
    df_stage2.to_csv(OUT_DIR / "eval_stage2_fetchk_rerankk_ho.csv", index=False)
    best2 = get_best_row(df_stage2, sort_col=HO_SORT_COL)
    print(f"[ho 2단계 최적] fetch_k={best2['fetch_k']}, rerank_k={best2['rerank_k']} -> MRR={best2['MRR']}")

    combos_stage3 = [(False, None)] + [(True, t) for t in MAX_CROSS_REF_TOKENS_GRID]
    df_stage3_rows = []
    print(f"\n━━━ [3단계 use_cross_refs×max_cross_ref_tokens (reranker ON)] ho — {len(combos_stage3)}개 조합 ━━━")
    for i, (use_xref, max_tok) in enumerate(combos_stage3, 1):
        params = dict(
            alpha=best1["alpha"], rrf_k=best1["rrf_k"],
            fetch_k=best2["fetch_k"], rerank_k=best2["rerank_k"], use_reranker=True,
            use_cross_refs=use_xref, max_cross_ref_tokens=max_tok,
        )
        print(f"  [{i}/{len(combos_stage3)}] use_cross_refs={use_xref} max_cross_ref_tokens={max_tok} ...", end="")
        result = evaluate_params("ho", client, model, laws_ref, reranker, gold_standard, cache, params)
        df_stage3_rows.append(result)
        print(f" -> MRR={result['MRR']} Recall@10={result['Recall@10']} ({result['avg_sec_per_query']}초/쿼리)")

    df_stage3 = pd.DataFrame(df_stage3_rows).sort_values("MRR", ascending=False)
    df_stage3.to_csv(OUT_DIR / "eval_stage3_crossref_ho.csv", index=False)
    best3 = get_best_row(df_stage3, sort_col=HO_SORT_COL)
    print(f"[ho 3단계 최적] use_cross_refs={best3['use_cross_refs']}, "
          f"max_cross_ref_tokens={best3['max_cross_ref_tokens']} -> MRR={best3['MRR']}")

    final = dict(
        variant="ho", use_reranker=True,
        alpha=best1["alpha"], rrf_k=best1["rrf_k"],
        fetch_k=best2["fetch_k"], rerank_k=best2["rerank_k"],
        use_cross_refs=best3["use_cross_refs"], max_cross_ref_tokens=best3["max_cross_ref_tokens"],
        MRR=best3["MRR"],
        **{f"Recall@{k}": best3[f"Recall@{k}"] for k in RECALL_AT_KS},
        **{f"FullCoverage@{k}": best3[f"FullCoverage@{k}"] for k in RECALL_AT_KS},
    )
    return final


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# HoRAG 단계적 sweep — reranker OFF (신규)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def sweep_ho_off(client, model, laws_ref, reranker, gold_standard, cache) -> dict:
    """
    jo와 마찬가지로 alpha×rrf_k, fetch_k를 reranker OFF 기준으로 처음부터
    다시 찾는다. 3단계(cross-ref)는 reranker ON과 동일한 그리드로 sweep —
    max_cross_ref_tokens 트리밍은 리랭커의 tokenizer로 토큰 수만 세는
    거라 use_reranker=False에서도 그대로 동작한다 (점수 계산과는 무관).
    """
    df_stage1_off = run_grid(
        "ho", client, model, laws_ref, reranker, gold_standard, cache,
        fixed_params=dict(
            fetch_k=DEFAULT_FETCH_K, rerank_k=DEFAULT_RERANK_K, use_reranker=False,
            use_cross_refs=True, max_cross_ref_tokens=DEFAULT_MAX_CROSS_REF_TOKENS,
        ),
        grid_params=dict(alpha=ALPHA_GRID, rrf_k=RRF_K_GRID),
        stage_name="1단계_off alpha×rrf_k (reranker OFF)",
    )
    df_stage1_off.to_csv(OUT_DIR / "eval_stage1_off_alpha_rrf_ho.csv", index=False)
    best1 = get_best_row(df_stage1_off, sort_col=HO_SORT_COL)
    print(f"[ho 1단계_off 최적] alpha={best1['alpha']}, rrf_k={best1['rrf_k']} -> MRR={best1['MRR']}")

    df_stage2_rows = []
    print(f"\n━━━ [2단계_off fetch_k (reranker OFF)] ho — {len(FETCH_K_GRID)}개 조합 ━━━")
    for i, fk in enumerate(FETCH_K_GRID, 1):
        params = dict(
            alpha=best1["alpha"], rrf_k=best1["rrf_k"],
            fetch_k=fk, rerank_k=DEFAULT_RERANK_K, use_reranker=False,
            use_cross_refs=True, max_cross_ref_tokens=DEFAULT_MAX_CROSS_REF_TOKENS,
        )
        print(f"  [{i}/{len(FETCH_K_GRID)}] fetch_k={fk} ...", end="")
        result = evaluate_params("ho", client, model, laws_ref, reranker, gold_standard, cache, params)
        df_stage2_rows.append(result)
        print(f" -> MRR={result['MRR']} Recall@10={result['Recall@10']} ({result['avg_sec_per_query']}초/쿼리)")

    df_stage2_off = pd.DataFrame(df_stage2_rows).sort_values("MRR", ascending=False)
    df_stage2_off.to_csv(OUT_DIR / "eval_stage2_off_fetchk_ho.csv", index=False)
    best2 = get_best_row(df_stage2_off, sort_col=HO_SORT_COL)
    print(f"[ho 2단계_off 최적] fetch_k={best2['fetch_k']} -> MRR={best2['MRR']}")

    combos_stage3 = [(False, None)] + [(True, t) for t in MAX_CROSS_REF_TOKENS_GRID]
    df_stage3_rows = []
    print(f"\n━━━ [3단계_off use_cross_refs×max_cross_ref_tokens (reranker OFF)] ho — {len(combos_stage3)}개 조합 ━━━")
    for i, (use_xref, max_tok) in enumerate(combos_stage3, 1):
        params = dict(
            alpha=best1["alpha"], rrf_k=best1["rrf_k"],
            fetch_k=best2["fetch_k"], rerank_k=DEFAULT_RERANK_K, use_reranker=False,
            use_cross_refs=use_xref, max_cross_ref_tokens=max_tok,
        )
        print(f"  [{i}/{len(combos_stage3)}] use_cross_refs={use_xref} max_cross_ref_tokens={max_tok} ...", end="")
        result = evaluate_params("ho", client, model, laws_ref, reranker, gold_standard, cache, params)
        df_stage3_rows.append(result)
        print(f" -> MRR={result['MRR']} Recall@10={result['Recall@10']} ({result['avg_sec_per_query']}초/쿼리)")

    df_stage3_off = pd.DataFrame(df_stage3_rows).sort_values("MRR", ascending=False)
    df_stage3_off.to_csv(OUT_DIR / "eval_stage3_off_crossref_ho.csv", index=False)
    best3 = get_best_row(df_stage3_off, sort_col=HO_SORT_COL)
    print(f"[ho 3단계_off 최적] use_cross_refs={best3['use_cross_refs']}, "
          f"max_cross_ref_tokens={best3['max_cross_ref_tokens']} -> MRR={best3['MRR']}")

    final = dict(
        variant="ho", use_reranker=False,
        alpha=best1["alpha"], rrf_k=best1["rrf_k"],
        fetch_k=best2["fetch_k"], rerank_k=DEFAULT_RERANK_K,
        use_cross_refs=best3["use_cross_refs"], max_cross_ref_tokens=best3["max_cross_ref_tokens"],
        MRR=best3["MRR"],
        **{f"Recall@{k}": best3[f"Recall@{k}"] for k in RECALL_AT_KS},
        **{f"FullCoverage@{k}": best3[f"FullCoverage@{k}"] for k in RECALL_AT_KS},
    )
    return final


def main():
    client   = get_qdrant_client()
    model    = load_embed_model()
    laws_ref = load_laws_ref()
    gold_standard = load_gold_standard()

    print(f"실버 스탠다드 {len(gold_standard)}개 로드 완료")

    # reranker는 무거우니 한 번만 로드 (grid 안에서 껐다 켰다는 플래그로만 토글).
    # off 전용 sweep에서도 이 객체를 그대로 넘긴다 — cross-ref 토큰 트리밍이
    # tokenizer만 필요로 해서 use_reranker=False여도 정상 동작하기 때문.
    reranker = load_reranker(device=RERANKER_DEVICE)

    # 전체 단계에서 재사용할 캐시 — jo/ho 두 variant, on/off 모든 단계에서 공유.
    # (alpha/rrf_k/rerank_k는 raw_search 캐시와 무관하므로 on/off sweep끼리도
    #  같은 fetch_k만 맞으면 raw_search가 재사용된다.)
    # 세션이 끊겨도 이어받을 수 있게 디스크에서 불러오거나 없으면 새로 시작한다.
    cache = SweepCache.load_or_new(CACHE_PATH)

    t_start = time.time()

    try:
        print("\n" + "=" * 60)
        print("JoRAG 단계적 sweep 시작 (reranker ON)")
        print("=" * 60)
        final_jo_on = sweep_jo(client, model, laws_ref, reranker, gold_standard, cache)
        cache.save(CACHE_PATH)  # 단계 끝날 때마다 저장 — 다음 단계에서 끊겨도 여기까지는 보존됨

        print("\n" + "=" * 60)
        print("JoRAG 단계적 sweep 시작 (reranker OFF — 비교용)")
        print("=" * 60)
        final_jo_off = sweep_jo_off(client, model, laws_ref, reranker, gold_standard, cache)
        cache.save(CACHE_PATH)

        print("\n" + "=" * 60)
        print("HoRAG 단계적 sweep 시작 (reranker ON)")
        print("=" * 60)
        final_ho_on = sweep_ho(client, model, laws_ref, reranker, gold_standard, cache)
        cache.save(CACHE_PATH)

        print("\n" + "=" * 60)
        print("HoRAG 단계적 sweep 시작 (reranker OFF — 비교용)")
        print("=" * 60)
        final_ho_off = sweep_ho_off(client, model, laws_ref, reranker, gold_standard, cache)
    finally:
        # 중간에 예외가 나도(마지막 단계 저장 전이라도) 그때까지의 캐시는 남긴다.
        cache.save(CACHE_PATH)

    t_total = time.time() - t_start

    df_final = pd.DataFrame([final_jo_on, final_jo_off, final_ho_on, final_ho_off])
    df_final.to_csv(OUT_DIR / "eval_final_best.csv", index=False)

    print("\n" + "=" * 60)
    print("✅ 전체 단계적 sweep 완료 (jo/ho × reranker on/off)")
    print("=" * 60)
    print(f"⏱️  총 소요 시간: {t_total:.1f}초 ({t_total/60:.1f}분)")
    print(f"📦 최종 캐시 통계: {cache.stats()}")
    print("\n=== 최종 확정 조합 (cascaded는 yoonha_rag_eval_cascaded_staged.py에서 추가됨) ===")
    print(df_final.to_string(index=False))

    for variant, label in [("jo", "JoRAG"), ("ho", "HoRAG")]:
        on_mrr = df_final.loc[(df_final.variant == variant) & (df_final.use_reranker == True), "MRR"].iloc[0]
        off_mrr = df_final.loc[(df_final.variant == variant) & (df_final.use_reranker == False), "MRR"].iloc[0]
        print(f"\n[{label}] reranker 효과: MRR {off_mrr} (OFF) -> {on_mrr} (ON), 차이 {on_mrr - off_mrr:+.4f}")


if __name__ == "__main__":
    main()

Writing /content/yoonha_rag_eval_staged.py


In [11]:
import os

os.environ["QDRANT_LOCAL_PATH"] = "/content/qdrant_local"
os.environ["RERANKER_DEVICE"] = "cuda"

!python /content/yoonha_rag_eval_staged.py

🗄️  로컬(파일 기반) Qdrant 사용: /content/qdrant_local
📦 임베딩 모델 로드: BAAI/bge-m3
Fetching 30 files:   0% 0/30 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Fetching 30 files: 100% 30/30 [01:01<00:00,  2.05s/it]
Download complete: 100% 4.54G/4.54G [01:01<00:00, 73.6MB/s]
Loading weights: 100% 391/391 [00:04<00:00, 85.44it/s]
  ⚠️  laws_ref.json 없음: /data/hn_seed/law_refs.json
실버 스탠다드 100개 로드 완료
📦 Re-ranker 로드: BAAI/bge-reranker-v2-m3


## 6. 결과 확인

In [20]:
import pandas as pd

df_final = pd.read_csv("/content/eval_final_best.csv")
print("=== 최종 확정 조합 (jo/ho × reranker on/off, 4행) ===")
df_final


=== 최종 확정 조합 (jo/ho × reranker on/off, 4행) ===


,variant,use_reranker,alpha,rrf_k,fetch_k,rerank_k,MRR,Recall@1,Recall@2,Recall@3,...,Recall@10,FullCoverage@1,FullCoverage@2,FullCoverage@3,FullCoverage@5,FullCoverage@10,use_cross_refs,max_cross_ref_tokens,jo_top_k,stage1_recall
0,jo,True,0.3,20,80,20,0.8843,0.80,0.95,0.97,...,0.98,0.59,0.77,0.83,0.84,0.89,NaN,NaN,NaN,NaN
1,jo,False,1.0,100,20,10,0.7800,0.68,0.82,0.86,...,0.95,0.53,0.68,0.72,0.77,0.84,NaN,NaN,NaN,NaN
2,ho,True,0.3,20,30,30,0.8850,0.80,0.95,0.97,...,0.98,0.59,0.77,0.82,0.83,0.89,True,NaN,NaN,NaN
3,ho,False,1.0,20,20,10,0.7820,0.69,0.81,0.83,...,0.95,0.51,0.65,0.70,0.75,0.81,True,NaN,NaN,NaN
4,cascaded,True,0.3,20,30,30,0.8753,0.79,0.94,0.96,...,0.98,0.58,0.76,0.81,0.83,0.90,True,NaN,30.0,0.97
5,cascaded,False,1.0,20,20,10,0.7916,0.70,0.81,0.84,...,0.97,0.52,0.64,0.69,0.78,0.89,True,NaN,20.0,0.97


In [21]:
# 각 단계별 상세 결과도 확인 가능 (on / off 둘 다)
df_stage1_jo = pd.read_csv("/content/eval_stage1_alpha_rrf_jo.csv")
df_stage2_jo = pd.read_csv("/content/eval_stage2_fetchk_rerankk_jo.csv")
df_stage1_off_jo = pd.read_csv("/content/eval_stage1_off_alpha_rrf_jo.csv")
df_stage2_off_jo = pd.read_csv("/content/eval_stage2_off_fetchk_jo.csv")

df_stage1_ho = pd.read_csv("/content/eval_stage1_alpha_rrf_ho.csv")
df_stage2_ho = pd.read_csv("/content/eval_stage2_fetchk_rerankk_ho.csv")
df_stage3_ho = pd.read_csv("/content/eval_stage3_crossref_ho.csv")
df_stage1_off_ho = pd.read_csv("/content/eval_stage1_off_alpha_rrf_ho.csv")
df_stage2_off_ho = pd.read_csv("/content/eval_stage2_off_fetchk_ho.csv")
df_stage3_off_ho = pd.read_csv("/content/eval_stage3_off_crossref_ho.csv")

print("=== jo 1단계 ON (alpha×rrf_k) 상위 5개 ===")
display(df_stage1_jo.head())
print("=== jo 1단계 OFF (alpha×rrf_k) 상위 5개 ===")
display(df_stage1_off_jo.head())

print("=== jo 2단계 ON (fetch_k×rerank_k) 상위 5개 ===")
display(df_stage2_jo.head())
print("=== jo 2단계 OFF (fetch_k) 상위 5개 ===")
display(df_stage2_off_jo.head())

print("=== ho 1단계 ON (alpha×rrf_k) 상위 5개 ===")
display(df_stage1_ho.head())
print("=== ho 1단계 OFF (alpha×rrf_k) 상위 5개 ===")
display(df_stage1_off_ho.head())

print("=== ho 2단계 ON (fetch_k×rerank_k) 상위 5개 ===")
display(df_stage2_ho.head())
print("=== ho 2단계 OFF (fetch_k) 상위 5개 ===")
display(df_stage2_off_ho.head())

print("=== ho 3단계 ON (use_cross_refs×max_cross_ref_tokens) 상위 5개 ===")
display(df_stage3_ho.head())
print("=== ho 3단계 OFF (use_cross_refs×max_cross_ref_tokens) 상위 5개 ===")
display(df_stage3_off_ho.head())


=== jo 1단계 ON (alpha×rrf_k) 상위 5개 ===


,variant,fetch_k,rerank_k,use_reranker,alpha,rrf_k,Recall@1,FullCoverage@1,Recall@2,FullCoverage@2,Recall@3,FullCoverage@3,Recall@5,FullCoverage@5,Recall@10,FullCoverage@10,MRR,avg_sec_per_query
0,jo,50,10,True,0.3,20,0.8,0.59,0.95,0.77,0.97,0.83,0.98,0.84,0.98,0.89,0.8837,6.188
1,jo,50,10,True,0.3,40,0.8,0.59,0.95,0.77,0.97,0.83,0.98,0.84,0.98,0.89,0.8837,0.000
2,jo,50,10,True,0.3,60,0.8,0.59,0.95,0.77,0.97,0.83,0.98,0.84,0.98,0.89,0.8837,0.000
3,jo,50,10,True,0.3,100,0.8,0.59,0.95,0.77,0.97,0.83,0.98,0.84,0.98,0.89,0.8837,0.000
4,jo,50,10,True,0.5,20,0.8,0.59,0.95,0.77,0.97,0.83,0.98,0.84,0.98,0.89,0.8837,0.000


=== jo 1단계 OFF (alpha×rrf_k) 상위 5개 ===


,variant,fetch_k,rerank_k,use_reranker,alpha,rrf_k,Recall@1,FullCoverage@1,Recall@2,FullCoverage@2,Recall@3,FullCoverage@3,Recall@5,FullCoverage@5,Recall@10,FullCoverage@10,MRR,avg_sec_per_query
0,jo,50,10,False,1.0,100,0.68,0.53,0.82,0.68,0.86,0.72,0.88,0.77,0.95,0.84,0.78,0.0
1,jo,50,10,False,1.0,60,0.68,0.53,0.82,0.68,0.86,0.72,0.88,0.77,0.95,0.84,0.78,0.0
2,jo,50,10,False,1.0,40,0.68,0.53,0.82,0.68,0.86,0.72,0.88,0.77,0.95,0.84,0.78,0.0
3,jo,50,10,False,1.0,20,0.68,0.53,0.82,0.68,0.86,0.72,0.88,0.77,0.95,0.84,0.78,0.0
4,jo,50,10,False,0.7,20,0.66,0.52,0.82,0.66,0.86,0.72,0.88,0.77,0.95,0.84,0.77,0.0


=== jo 2단계 ON (fetch_k×rerank_k) 상위 5개 ===


...

,variant,alpha,rrf_k,fetch_k,rerank_k,use_reranker,use_cross_refs,max_cross_ref_tokens,Recall@1,FullCoverage@1,Recall@2,FullCoverage@2,Recall@3,FullCoverage@3,Recall@5,FullCoverage@5,Recall@10,FullCoverage@10,MRR,avg_sec_per_query
0,ho,1.0,20,20,10,False,True,NaN,0.69,0.51,0.81,0.65,0.83,0.7,0.89,0.75,0.95,0.81,0.7820,0.048
1,ho,1.0,20,30,10,False,True,NaN,0.69,0.51,0.81,0.65,0.83,0.7,0.89,0.75,0.95,0.81,0.7814,0.122
2,ho,1.0,20,50,10,False,True,NaN,0.69,0.51,0.81,0.65,0.83,0.7,0.89,0.75,0.95,0.81,0.7814,0.286
3,ho,1.0,20,80,10,False,True,NaN,0.69,0.51,0.81,0.65,0.83,0.7,0.89,0.75,0.95,0.81,0.7814,0.633


=== ho 3단계 ON (use_cross_refs×max_cross_ref_tokens) 상위 5개 ===


,variant,alpha,rrf_k,fetch_k,rerank_k,use_reranker,use_cross_refs,max_cross_ref_tokens,Recall@1,FullCoverage@1,Recall@2,FullCoverage@2,Recall@3,FullCoverage@3,Recall@5,FullCoverage@5,Recall@10,FullCoverage@10,MRR,avg_sec_per_query
0,ho,0.3,20,30,30,True,True,NaN,0.8,0.59,0.95,0.77,0.97,0.82,0.98,0.83,0.98,0.89,0.8850,0.125
1,ho,0.3,20,30,30,True,False,NaN,0.8,0.59,0.95,0.77,0.97,0.82,0.98,0.84,0.98,0.86,0.8837,0.000
2,ho,0.3,20,30,30,True,True,500.0,0.8,0.59,0.95,0.77,0.97,0.82,0.98,0.84,0.98,0.86,0.8837,0.199
3,ho,0.3,20,30,30,True,True,1000.0,0.8,0.59,0.95,0.77,0.97,0.82,0.98,0.84,0.98,0.86,0.8837,0.203
4,ho,0.3,20,30,30,True,True,2000.0,0.8,0.59,0.95,0.77,0.97,0.82,0.98,0.84,0.98,0.86,0.8837,0.204


=== ho 3단계 OFF (use_cross_refs×max_cross_ref_tokens) 상위 5개 ===


,variant,alpha,rrf_k,fetch_k,rerank_k,use_reranker,use_cross_refs,max_cross_ref_tokens,Recall@1,FullCoverage@1,Recall@2,FullCoverage@2,Recall@3,FullCoverage@3,Recall@5,FullCoverage@5,Recall@10,FullCoverage@10,MRR,avg_sec_per_query
0,ho,1.0,20,20,10,False,True,NaN,0.69,0.51,0.81,0.65,0.83,0.7,0.89,0.75,0.95,0.81,0.7820,0.047
1,ho,1.0,20,20,10,False,False,NaN,0.69,0.51,0.81,0.65,0.83,0.7,0.89,0.75,0.95,0.81,0.7814,0.000
2,ho,1.0,20,20,10,False,True,500.0,0.69,0.51,0.81,0.65,0.83,0.7,0.89,0.75,0.95,0.81,0.7814,0.099
3,ho,1.0,20,20,10,False,True,1000.0,0.69,0.51,0.81,0.65,0.83,0.7,0.89,0.75,0.95,0.81,0.7814,0.102
4,ho,1.0,20,20,10,False,True,2000.0,0.69,0.51,0.81,0.65,0.83,0.7,0.89,0.75,0.95,0.81,0.7814,0.104


## 7. 계층형(Cascaded) 파이프라인 추가 — 조랭크(JoRAG) 이후 호랭크(HoRAG) 연결

지금까지는 JoRAG/HoRAG를 완전히 독립적으로(병렬로) 검색했습니다. 여기서는
**1단계로 JoRAG(조 단위)를 먼저 넓게 검색해서 후보 조를 추리고, 2단계로 그
조들 안에서만 HoRAG(호/목/세목 단위)를 검색**하는 계층형(coarse-to-fine)
파이프라인을 단일 리랭커 버전에 추가합니다.

- 리랭커가 이제 `bge-reranker-v2-m3` 하나뿐이라, 구버전 계층형(리랭커 2개를
  1단계 직후/parent fetch 이후로 나눠 걸었던 구조)과 달리 **병렬 HoRAG와
  동일하게 parent fetch + cross-ref 트리밍 이후 리랭커를 1번만** 적용합니다.
- 1단계 필터링에 쓰는 `jo_top_k`만 새로운 축이고, 나머지(alpha/rrf_k/fetch_k/
  rerank_k/use_cross_refs/max_cross_ref_tokens)는 이미 병렬 HoRAG sweep으로
  확정한 값을 그대로 재사용합니다.
- `_hybrid_search`를 필터 없이 그대로 재사용하기 때문에, jo/ho 병렬 버전의
  raw_search 캐시와 그대로 공유됩니다 (jo_top_k를 여러 개 sweep해도 Qdrant
  재검색이 추가로 발생하지 않음).


In [12]:
%%writefile /content/yoonha_law_rag_cascaded.py
"""
Workit - 계층형(Cascaded) RAG 파이프라인, reranker 단일화 버전
파일명: rag/yoonha_law_rag_cascaded.py

병렬 버전(yoonha_law_rag.py의 JoRAG/HoRAG 독립 검색)과는 별개로 관리되는
계층형(coarse-to-fine) variant. "조랭크(JoRAG) 이후 호랭크(HoRAG)를 연결"하는
구조 — 1단계에서 넓게 candidate 조를 추린 뒤, 2단계에서 그 조 안에서만
호/목/세목 단위로 좁혀서 검색한다.

흐름:
  1단계 (coarse) : law_kb_jo에서 조 단위로 넓게 검색해 상위
                    jo_top_k개 조만 후보로 추린다 (JoRAG와 동일한 하이브리드
                    검색 로직 재사용 — _hybrid_search를 그대로 import).
  2단계 (fine)   : law_kb_ho에서 검색한 뒤, 각 후보의 chunk_id를
                    derive_jo_id()로 조 단위 id로 역산해서 1단계 후보 조
                    안에 있는 것만 Python 사이드에서 남긴다.
  3단계          : (옵션) cross_refs 확장 — 명시적 인용은 1단계 필터 밖이어도
                    살려서 확장한다 (조 단위 후보에 안 걸렸다고 인용 관계까지
                    끊어버리면 너무 공격적인 필터링이라고 판단).
  4단계          : parent fetch로 최종 출력을 조 텍스트로 통일.
  5단계          : (옵션) cross-ref로 추가된 후보만 토큰 예산 기준으로 트리밍
                    (병렬 HoRAG와 동일한 _trim_cross_refs_by_token_budget 재사용).
  6단계          : reranker(bge-reranker-v2-m3) 1회만 적용.

이전 버전(리랭커 2개짜리 cascaded, yoonha_law_rag_cascaded.py 구버전)과의 차이:
  - reranker1(1단계 직후)/reranker2(parent fetch 이후) 이원 구조를 걷어내고,
    병렬 HoRAG(_search_ho)와 동일하게 "parent fetch + cross-ref 트리밍 이후"
    reranker를 1번만 적용한다. 리랭커가 하나뿐이라 굳이 2번 나눠 돌릴 이유가
    없고, v3 문서(3-2a)에서 확인됐듯 reranker를 조 전체 텍스트 확정 이후에
    돌리는 쪽이 오히려 더 안정적이었다(HoRAG r1+r2 조합이 r1 단독보다 나빴던
    사례 참고).
  - max_cross_ref_tokens 토큰 예산 트리밍(병렬 HoRAG에 이번에 추가된 기능)을
    캐스케이드에도 동일하게 적용했다.

주의:
  - 1단계 recall이 2단계 recall의 상한선이다. 1단계에서 정답 조가 top-k
    밖으로 밀리면 2단계에서 아무리 잘 검색해도 복구 불가능하다. 그래서
    jo_top_k는 반드시 sweep 대상에 넣어야 하고, 이 모듈은 "1단계에서
    정답 조가 살아남았는가"를 별도로 진단할 수 있는 get_stage1_jo_candidates도
    노출한다 (yoonha_rag_eval_cascaded_staged.py에서 stage1_recall 지표로 사용).
  - PYG(예규)는 조가 없어 jo-level이 항 단위로 anchor된다. derive_jo_id()가
    PYG는 앞 5토큰을, 일반 법령은 앞 4토큰을 jo_id로 역산하도록 이미 분기
    처리돼 있어서 필터링 자체는 정상 동작한다.
  - 이 모듈은 yoonha_law_rag.py의 함수 이름을 재사용하지 않는다. 공용 유틸
    (get_vectors, CrossEncoderReranker, SweepCache, LawRef, ClauseResult,
    chunk_contract, _hybrid_search, _rerank, _fetch_parent_texts,
    _expand_with_cross_refs, _trim_cross_refs_by_token_budget, _build_law_refs)은
    그대로 import해서 쓰고, 새로 정의하는 함수는 전부 *_cascaded 접미사를 붙였다.
  - 최종 출력 chunk_id는 (병렬 HoRAG와 동일하게) ho-level chunk_id다 — 정답
    비교 시 gold_standard의 relevant_docs_ho와 같은 필드를 기준으로 삼는다.
"""

from __future__ import annotations

from qdrant_client import QdrantClient
from FlagEmbedding import BGEM3FlagModel

from yoonha_law_rag import (
    COLLECTION_JO,
    COLLECTION_HO,
    DEFAULT_FETCH_K,
    DEFAULT_RERANK_K,
    DEFAULT_TOP_K,
    DEFAULT_ALPHA,
    DEFAULT_RRF_K,
    DEFAULT_MAX_CROSS_REF_TOKENS,
    CrossEncoderReranker,
    LawRef,
    ClauseResult,
    SweepCache,
    chunk_contract,
    get_vectors,
    derive_jo_id,
    load_laws_ref,
    load_embed_model,
    load_reranker,
    _hybrid_search,
    _rerank,
    _fetch_parent_texts,
    _expand_with_cross_refs,
    _trim_cross_refs_by_token_budget,
    _build_law_refs,
)

# 계층형 전용 기본값 — 1단계(조 단위)에서 몇 개 조를 후보로 남길지.
# 너무 작으면 1단계에서 정답 조가 걸러져서 2단계가 복구 불가능해지고,
# 너무 크면 사실상 병렬 HoRAG와 차이가 없어진다. sweep으로 찾아야 한다.
DEFAULT_JO_TOP_K = 20


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 1단계: 조 단위 후보 추출
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def get_stage1_jo_candidates(
    clause_text: str,
    client     : QdrantClient,
    model      : BGEM3FlagModel,
    alpha      : float = DEFAULT_ALPHA,
    fetch_k    : int   = DEFAULT_FETCH_K,
    rrf_k      : int   = DEFAULT_RRF_K,
    jo_top_k   : int   = DEFAULT_JO_TOP_K,
    cache      : SweepCache | None = None,
) -> list[str]:
    """
    1단계: law_kb_jo에서 조 단위로 넓게 검색해 상위 jo_top_k개
    조의 chunk_id만 반환한다. _hybrid_search를 그대로 재사용하므로
    JoRAG(병렬 버전)의 raw_search 캐시와도 공유된다.
    """
    candidates = _hybrid_search(clause_text, client, model, COLLECTION_JO, fetch_k, alpha, rrf_k, cache)
    return [c["chunk_id"] for c in candidates[:jo_top_k]]


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 2단계: jo_id로 제한된 ho 후보 필터링 (Python 사이드)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _hybrid_search_ho_filtered(
    clause_text       : str,
    client            : QdrantClient,
    model             : BGEM3FlagModel,
    fetch_k           : int,
    alpha             : float,
    rrf_k             : int,
    allowed_parent_ids: list[str],
    cache             : SweepCache | None = None,
) -> list[dict]:
    """
    2단계: law_kb_ho에서 검색한 뒤, 각 후보의 chunk_id를 derive_jo_id()로
    조 단위 id로 역산해서 allowed_parent_ids(1단계 후보 조) 안에 있는 것만 남긴다.

    payload의 parent_chunk_id 필드는 HO 컬렉션 자기 자신을 가리키는 값이라
    이 용도로 쓸 수 없다 (derive_jo_id 참고) — chunk_id 문자열 자체에서
    역산하는 방식만 신뢰한다.

    _hybrid_search를 필터 없이 그대로 재사용하므로, jo_top_k가 달라져도
    raw_search 캐시가 (collection, clause_text, fetch_k) 키 하나로 공유된다
    (병렬 HoRAG의 raw_search 캐시와도 공유됨 — 사실상 공짜).
    """
    candidates = _hybrid_search(clause_text, client, model, COLLECTION_HO, fetch_k, alpha, rrf_k, cache)
    allowed_set = set(allowed_parent_ids)
    return [c for c in candidates if derive_jo_id(c["chunk_id"]) in allowed_set]


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 전체 계층형 검색
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _search_cascaded(
    clause_text  : str,
    client       : QdrantClient,
    model        : BGEM3FlagModel,
    laws_ref     : dict[str, dict],
    reranker     : CrossEncoderReranker | None = None,
    use_reranker : bool  = False,
    use_cross_refs: bool = True,
    max_cross_ref_tokens: int | None = DEFAULT_MAX_CROSS_REF_TOKENS,
    top_k        : int   = DEFAULT_TOP_K,
    alpha        : float = DEFAULT_ALPHA,
    fetch_k      : int   = DEFAULT_FETCH_K,
    rerank_k     : int   = DEFAULT_RERANK_K,
    rrf_k        : int   = DEFAULT_RRF_K,
    jo_top_k     : int   = DEFAULT_JO_TOP_K,
    cache        : SweepCache | None = None,
) -> list[LawRef]:
    """
    1단계(JoRAG top jo_top_k) → 2단계(그 조들로 제한된 HoRAG) → cross-ref 확장
    → parent fetch(조 텍스트 통일) → 토큰 예산 트리밍 → reranker(1회) 순서로
    진행하는 계층형 검색. 병렬 HoRAG(_search_ho)와 구조가 거의 동일하고,
    차이는 맨 앞에 "1단계 조 후보로 좁히는" 필터 한 단계가 추가된 것뿐이다.
    """
    allowed_parent_ids = get_stage1_jo_candidates(
        clause_text, client, model, alpha, fetch_k, rrf_k, jo_top_k, cache,
    )

    if not allowed_parent_ids:
        return []

    candidates = _hybrid_search_ho_filtered(
        clause_text, client, model, fetch_k, alpha, rrf_k, allowed_parent_ids, cache,
    )

    if use_cross_refs:
        candidates = _expand_with_cross_refs(candidates, client, cache)

    # parent fetch: 호/목/세목 → 조 텍스트로 교체 (최종 출력 단위 통일, 병렬 버전과 동일)
    candidates = _fetch_parent_texts(candidates, client, parent_collection=COLLECTION_JO, cache=cache)

    if use_cross_refs and max_cross_ref_tokens is not None:
        if reranker is not None:
            candidates = _trim_cross_refs_by_token_budget(candidates, reranker, max_cross_ref_tokens)
        else:
            print("  ⚠️  max_cross_ref_tokens가 설정됐지만 reranker가 없어 토큰 트리밍을 건너뜁니다.")

    if use_reranker and reranker and candidates:
        candidates = _rerank(clause_text, candidates, reranker, rerank_k, "cascaded", cache)

    return _build_law_refs(candidates, laws_ref, top_k, with_xref=use_cross_refs)


def review_contract_cascaded(
    contract_text: str,
    client       : QdrantClient,
    model        : BGEM3FlagModel,
    laws_ref     : dict[str, dict] | None = None,
    reranker     : CrossEncoderReranker | None = None,
    use_reranker : bool  = False,
    use_cross_refs: bool = True,
    max_cross_ref_tokens: int | None = DEFAULT_MAX_CROSS_REF_TOKENS,
    top_k        : int   = DEFAULT_TOP_K,
    alpha        : float = DEFAULT_ALPHA,
    fetch_k      : int   = DEFAULT_FETCH_K,
    rerank_k     : int   = DEFAULT_RERANK_K,
    rrf_k        : int   = DEFAULT_RRF_K,
    jo_top_k     : int   = DEFAULT_JO_TOP_K,
    cache        : SweepCache | None = None,
) -> list[ClauseResult]:
    """계층형 RAG 메인 인터페이스 (review_contract_jo/ho와 동일한 사용 패턴)."""
    if laws_ref is None:
        laws_ref = load_laws_ref()

    clauses = chunk_contract(contract_text)
    results : list[ClauseResult] = []
    print(f"[Cascaded] 총 {len(clauses)}개 청크 검색 중...")

    for i, clause in enumerate(clauses, 1):
        print(f"  [{i}/{len(clauses)}] {clause['clause_number']} ...", end="\r")

        law_refs = _search_cascaded(
            clause["clause_text"], client, model, laws_ref,
            reranker, use_reranker, use_cross_refs, max_cross_ref_tokens,
            top_k, alpha, fetch_k, rerank_k, rrf_k, jo_top_k, cache,
        )
        categories = list(dict.fromkeys(r.category for r in law_refs if r.category))

        results.append(ClauseResult(
            clause_number=clause["clause_number"],
            clause_text  =clause["clause_text"],
            law_refs     =law_refs,
            categories   =categories,
        ))

    print(f"\n[Cascaded] ✅ 완료")
    return results


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 편의: 단일 조항 검색 (sweep 스크립트에서 직접 호출)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def search_cascaded(clause_text: str, client: QdrantClient, model: BGEM3FlagModel,
                     laws_ref: dict, reranker=None, use_reranker=False,
                     use_cross_refs=True, max_cross_ref_tokens=DEFAULT_MAX_CROSS_REF_TOKENS,
                     top_k=DEFAULT_TOP_K, alpha=DEFAULT_ALPHA, fetch_k=DEFAULT_FETCH_K,
                     rerank_k=DEFAULT_RERANK_K, rrf_k=DEFAULT_RRF_K,
                     jo_top_k=DEFAULT_JO_TOP_K,
                     cache: SweepCache | None = None) -> list[LawRef]:
    return _search_cascaded(clause_text, client, model, laws_ref, reranker,
                             use_reranker, use_cross_refs, max_cross_ref_tokens,
                             top_k, alpha, fetch_k, rerank_k, rrf_k, jo_top_k, cache)


Writing /content/yoonha_law_rag_cascaded.py


## 8. jo_top_k 단계적 sweep 실행 (reranker ON/OFF 둘 다)

병렬 HoRAG의 확정 조합을 reranker ON/OFF 각각(`eval_final_best.csv`의
`ho` 행 2개) 그대로 고정하고, 계층형 전용 파라미터인 `jo_top_k`만
`[10, 20, 30, 40, 50]`로 sweep합니다 — cascaded-on / cascaded-off 두 번
돌아서 `eval_final_best.csv`에 `cascaded` 행 2개가 추가됩니다 (최종 6행:
jo/ho/cascaded × on/off).

이 셀을 실행하기 전에 **5번 섹션(단계적 sweep)이 먼저 끝나 있어야** 합니다
(`eval_final_best.csv`에 `ho` 행 2개(on/off)가 있어야 여기서 그 설정을
읽어옵니다).

`stage1_recall`도 같이 기록해서, 결과가 안 좋게 나왔을 때 "1단계에서 정답
조 자체가 걸러진 것"인지 "2단계 랭킹 자체의 문제"인지 구분할 수 있게 했습니다.


In [13]:
%%writefile /content/yoonha_rag_eval_cascaded_staged.py
"""
Workit - 계층형(Cascaded) RAG jo_top_k 단계적 sweep, reranker 단일화 버전
파일명: rag/yoonha_rag_eval_cascaded_staged.py

목적:
  yoonha_rag_eval_staged.py(병렬 JoRAG/HoRAG, on/off 둘 다)에서 이미 확정한
  최적 조합을 그대로 가져와 고정하고, 계층형 전용 파라미터인 jo_top_k만
  sweep한다. reranker ON/OFF 각각에 대해 대응하는 ho 설정(ho-on, ho-off)을
  그대로 재사용해서 cascaded-on / cascaded-off 두 번을 sweep한다 — "리랭커가
  계층형 구조에서도 얼마나 기여하는지"를 병렬 HoRAG와 동일한 방식으로 비교하기
  위함이다.

  다른 축(alpha, rrf_k, fetch_k, use_cross_refs, max_cross_ref_tokens)을 전부
  고정하는 이유는 1) 이미 병렬 HoRAG sweep(on/off 각각)으로 최적 hybrid/reranker
  설정을 찾아뒀고 2) jo_top_k는 "1단계 필터링이 얼마나 넓은지"만 결정하는
  독립적인 축이라 alpha/fetch_k 등과 교차로 다시 sweep할 필요가 낮기 때문이다.

stage1_recall 진단:
  jo_top_k가 작을수록 1단계에서 정답 조 자체가 후보에서 탈락할 위험이
  커진다. 이 탈락은 2단계에서 절대 복구되지 않으므로, "최소 1개의 정답
  ho 문서의 조(jo) 부모가 1단계 후보 안에 살아있는 쿼리의 비율"을
  stage1_recall로 따로 기록한다. reranker on/off와 무관하게 1단계는 항상
  RRF 점수 기준이라 stage1_recall 자체는 on/off 사이에 큰 차이가 없을
  것으로 예상되지만, 혹시 다를 경우를 대비해 둘 다 따로 기록한다.

출력:
  - eval_stage4_jotopk_cascaded.csv  (jo_top_k × reranker on/off별 상세 결과)
  - eval_final_best.csv 갱신          (variant="cascaded" 행 2개 추가/교체:
                                        use_reranker=True/False 각각 1행씩)

사용법:
    python rag/yoonha_rag_eval_cascaded_staged.py
"""

from __future__ import annotations

import json
import os
import time
from pathlib import Path

import pandas as pd
from qdrant_client import QdrantClient

from yoonha_law_rag import (
    load_embed_model,
    load_laws_ref,
    load_reranker,
    derive_jo_id,
    SweepCache,
    QDRANT_HOST,
    QDRANT_PORT,
)
from yoonha_law_rag_cascaded import (
    search_cascaded,
    get_stage1_jo_candidates,
    DEFAULT_JO_TOP_K,
)

QDRANT_LOCAL_PATH_ENV = os.environ.get("QDRANT_LOCAL_PATH")
RERANKER_DEVICE = os.environ.get("RERANKER_DEVICE", "cpu")


def get_qdrant_client() -> QdrantClient:
    if QDRANT_LOCAL_PATH_ENV:
        print(f"🗄️  로컬(파일 기반) Qdrant 사용: {QDRANT_LOCAL_PATH_ENV}")
        return QdrantClient(path=QDRANT_LOCAL_PATH_ENV)
    print(f"🗄️  서버 Qdrant 사용: {QDRANT_HOST}:{QDRANT_PORT}")
    return QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)


GOLD_STANDARD_PATH = Path("/content/eval_set.json")
OUT_DIR = Path("/content")
FINAL_BEST_PATH = OUT_DIR / "eval_final_best.csv"
CACHE_PATH = OUT_DIR / "sweep_cache_cascaded.pkl"  # 세션 끊겨도 이어받을 수 있게 디스크에 저장

# chunk_id 자체는 ho-level이지만 LLM에는 parent fetch된 조 텍스트가 최종
# 전달되므로, jo/ho와 동일하게 GT는 relevant_docs_jo(조 단위)로 통일한다.
GT_FIELD = "relevant_docs_jo"

RECALL_AT_KS = [1, 2, 3, 5, 10]
TOP_K_EVAL   = max(RECALL_AT_KS)

# ho/jo eval과 동일한 이유로, raw 후보를 넉넉히 뽑아 조 단위로 접은 뒤 자른다.
RAW_TOP_K_FOR_JO_DEDUP = 30

# jo_top_k sweep grid — v3 계층형 실험(4-1)과 동일한 범위를 그대로 사용.
JO_TOP_K_GRID = [10, 20, 30, 40, 50]


def load_gold_standard(path: Path = GOLD_STANDARD_PATH) -> list[dict]:
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def load_ho_best_config(use_reranker: bool) -> dict:
    """
    eval_final_best.csv에서 variant='ho' AND use_reranker=<주어진 값> 행을 읽어
    alpha/rrf_k/fetch_k/rerank_k/use_cross_refs/max_cross_ref_tokens을 그대로
    가져온다. 이 파일은 yoonha_rag_eval_staged.py를 먼저 돌려야 생긴다
    (jo/ho 각각 on/off 2행씩, 총 4행이 있어야 함).
    """
    if not FINAL_BEST_PATH.exists():
        raise FileNotFoundError(
            f"{FINAL_BEST_PATH} 가 없습니다. 먼저 yoonha_rag_eval_staged.py를 실행해서 "
            "병렬 JoRAG/HoRAG 최적 조합(on/off 둘 다)을 확정하세요."
        )
    df = pd.read_csv(FINAL_BEST_PATH)
    rows = df[(df["variant"] == "ho") & (df["use_reranker"] == use_reranker)]
    if rows.empty:
        raise ValueError(
            f"{FINAL_BEST_PATH} 안에 variant='ho', use_reranker={use_reranker} 행이 없습니다."
        )
    row = rows.iloc[0]

    max_tok = row["max_cross_ref_tokens"]
    max_tok = None if pd.isna(max_tok) else int(max_tok)

    return dict(
        alpha=float(row["alpha"]),
        rrf_k=int(row["rrf_k"]),
        fetch_k=int(row["fetch_k"]),
        rerank_k=int(row["rerank_k"]),
        use_cross_refs=bool(row["use_cross_refs"]),
        max_cross_ref_tokens=max_tok,
    )


def evaluate_jo_top_k(
    client, model, laws_ref, reranker, gold_standard, cache,
    jo_top_k: int, use_reranker: bool, base_params: dict,
) -> dict:
    recall_hits = {k: 0 for k in RECALL_AT_KS}
    full_coverage_hits = {k: 0 for k in RECALL_AT_KS}
    mrr = 0.0
    stage1_recall_hits = 0
    n = len(gold_standard)
    t0 = time.time()

    for item in gold_standard:
        gt_docs = set(item[GT_FIELD])
        gt_jo_ids = {derive_jo_id(d) for d in gt_docs}

        # 1단계 진단: gt 문서 중 최소 1개의 조 부모가 1단계 후보 안에 살아있는가
        allowed_parent_ids = get_stage1_jo_candidates(
            item["query"], client, model,
            alpha=base_params["alpha"], fetch_k=base_params["fetch_k"],
            rrf_k=base_params["rrf_k"], jo_top_k=jo_top_k, cache=cache,
        )
        if gt_jo_ids & set(allowed_parent_ids):
            stage1_recall_hits += 1

        law_refs = search_cascaded(
            clause_text=item["query"],
            client=client, model=model, laws_ref=laws_ref,
            reranker=reranker, use_reranker=use_reranker,
            top_k=RAW_TOP_K_FOR_JO_DEDUP, jo_top_k=jo_top_k, cache=cache,
            **base_params,
        )
        raw_ranked_ids = [r.chunk_id for r in law_refs]

        # 조 id로 접고, 순서를 유지한 채 중복만 제거 (jo/ho eval과 동일 로직)
        seen: set[str] = set()
        ranked_ids: list[str] = []
        for cid in raw_ranked_ids:
            jid = derive_jo_id(cid)
            if jid not in seen:
                seen.add(jid)
                ranked_ids.append(jid)

        for k in RECALL_AT_KS:
            top_k_set = set(ranked_ids[:k])
            if gt_docs & top_k_set:
                recall_hits[k] += 1
            if gt_docs and gt_docs.issubset(top_k_set):
                full_coverage_hits[k] += 1
        for rank, d in enumerate(ranked_ids, 1):
            if d in gt_docs:
                mrr += 1 / rank
                break

    elapsed = time.time() - t0
    result = {"variant": "cascaded", "use_reranker": use_reranker, "jo_top_k": jo_top_k, **base_params}
    for k in RECALL_AT_KS:
        result[f"Recall@{k}"] = round(recall_hits[k] / n, 4)
        result[f"FullCoverage@{k}"] = round(full_coverage_hits[k] / n, 4)
    result["MRR"] = round(mrr / n, 4)
    result["stage1_recall"] = round(stage1_recall_hits / n, 4)
    result["avg_sec_per_query"] = round(elapsed / n, 3)
    return result


def sweep_cascaded_for(
    use_reranker: bool, client, model, laws_ref, reranker, gold_standard, cache,
) -> tuple[pd.DataFrame, dict]:
    label = "ON" if use_reranker else "OFF"
    base_params = load_ho_best_config(use_reranker=use_reranker)
    print(f"\n[병렬 HoRAG 최적 설정 재사용 — reranker {label}] {base_params}")

    print(f"\n━━━ [jo_top_k sweep, reranker {label}] cascaded — {len(JO_TOP_K_GRID)}개 조합 ━━━")
    rows = []
    for i, jo_top_k in enumerate(JO_TOP_K_GRID, 1):
        print(f"  [{i}/{len(JO_TOP_K_GRID)}] jo_top_k={jo_top_k} ...", end="")
        result = evaluate_jo_top_k(
            client, model, laws_ref, reranker, gold_standard, cache,
            jo_top_k, use_reranker, base_params,
        )
        rows.append(result)
        print(f" -> MRR={result['MRR']} Recall@10={result['Recall@10']} "
              f"stage1_recall={result['stage1_recall']} ({result['avg_sec_per_query']}초/쿼리)")

    df = pd.DataFrame(rows).sort_values("MRR", ascending=False)
    best = df.iloc[0].to_dict()
    print(f"\n[cascaded {label} 최적] jo_top_k={best['jo_top_k']} -> MRR={best['MRR']} "
          f"(stage1_recall={best['stage1_recall']})")

    final_row = {
        "variant": "cascaded",
        "use_reranker": use_reranker,
        "alpha": base_params["alpha"],
        "rrf_k": base_params["rrf_k"],
        "fetch_k": base_params["fetch_k"],
        "rerank_k": base_params["rerank_k"],
        "MRR": best["MRR"],
        **{f"Recall@{k}": best[f"Recall@{k}"] for k in RECALL_AT_KS},
        **{f"FullCoverage@{k}": best[f"FullCoverage@{k}"] for k in RECALL_AT_KS},
        "use_cross_refs": base_params["use_cross_refs"],
        "max_cross_ref_tokens": base_params["max_cross_ref_tokens"],
        "jo_top_k": best["jo_top_k"],
        "stage1_recall": best["stage1_recall"],
    }
    return df, final_row


def upsert_final_best(new_rows: list[dict]) -> pd.DataFrame:
    """eval_final_best.csv에 variant='cascaded' 행들을 추가(이미 있으면 교체)한다."""
    df = pd.read_csv(FINAL_BEST_PATH)
    df = df[df["variant"] != "cascaded"]
    df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
    df.to_csv(FINAL_BEST_PATH, index=False)
    return df


def main():
    client   = get_qdrant_client()
    model    = load_embed_model()
    laws_ref = load_laws_ref()
    gold_standard = load_gold_standard()
    print(f"실버 스탠다드 {len(gold_standard)}개 로드 완료")

    reranker = load_reranker(device=RERANKER_DEVICE)
    cache = SweepCache.load_or_new(CACHE_PATH)

    try:
        df_on, final_on = sweep_cascaded_for(True, client, model, laws_ref, reranker, gold_standard, cache)
        cache.save(CACHE_PATH)
        df_off, final_off = sweep_cascaded_for(False, client, model, laws_ref, reranker, gold_standard, cache)
    finally:
        cache.save(CACHE_PATH)

    df_all = pd.concat([df_on, df_off], ignore_index=True).sort_values(
        ["use_reranker", "MRR"], ascending=[False, False]
    )
    df_all.to_csv(OUT_DIR / "eval_stage4_jotopk_cascaded.csv", index=False)

    df_final = upsert_final_best([final_on, final_off])

    ho_on_mrr = df_final.loc[(df_final.variant == "ho") & (df_final.use_reranker == True), "MRR"].iloc[0]
    ho_off_mrr = df_final.loc[(df_final.variant == "ho") & (df_final.use_reranker == False), "MRR"].iloc[0]

    print("\n" + "=" * 60)
    print("✅ 계층형(Cascaded) jo_top_k sweep 완료 (reranker on/off 둘 다)")
    print("=" * 60)
    print(f"[reranker ON]  병렬 HoRAG MRR: {ho_on_mrr}  |  Cascaded 최고 MRR: {final_on['MRR']}  "
          f"|  차이: {final_on['MRR'] - ho_on_mrr:+.4f}")
    print(f"[reranker OFF] 병렬 HoRAG MRR: {ho_off_mrr}  |  Cascaded 최고 MRR: {final_off['MRR']}  "
          f"|  차이: {final_off['MRR'] - ho_off_mrr:+.4f}")
    print(f"\nCascaded 자체의 reranker 효과: MRR {final_off['MRR']} (OFF) -> {final_on['MRR']} (ON), "
          f"차이 {final_on['MRR'] - final_off['MRR']:+.4f}")

    for label, best in [("ON", final_on), ("OFF", final_off)]:
        if best["stage1_recall"] < 0.95:
            print(f"⚠️  [reranker {label}] stage1_recall={best['stage1_recall']} — jo_top_k를 더 키우면 "
                  f"개선될 여지가 있습니다.")

    print("\n=== eval_final_best.csv (갱신됨, 총 6행: jo/ho/cascaded × on/off) ===")
    print(df_final.to_string(index=False))


if __name__ == "__main__":
    main()


Writing /content/yoonha_rag_eval_cascaded_staged.py


In [14]:
import os

os.environ["QDRANT_LOCAL_PATH"] = "/content/qdrant_local"
os.environ["RERANKER_DEVICE"] = "cuda"

!python /content/yoonha_rag_eval_cascaded_staged.py


🗄️  로컬(파일 기반) Qdrant 사용: /content/qdrant_local
📦 임베딩 모델 로드: BAAI/bge-m3
Fetching 30 files: 100% 30/30 [00:00<00:00, 4614.37it/s]
Download complete: : 0.00B [00:00, ?B/s]
Loading weights: 100% 391/391 [00:09<00:00, 41.83it/s]
  ⚠️  laws_ref.json 없음: /data/hn_seed/law_refs.json
실버 스탠다드 100개 로드 완료
📦 Re-ranker 로드: BAAI/bge-reranker-v2-m3
Loading weights: 100% 393/393 [00:00<00:00, 748.27it/s]
  💾 캐시 파일 없음, 새로 시작: /content/sweep_cache_cascaded.pkl

[병렬 HoRAG 최적 설정 재사용 — reranker ON] {'alpha': 0.3, 'rrf_k': 20, 'fetch_k': 30, 'rerank_k': 30, 'use_cross_refs': True, 'max_cross_ref_tokens': None}

━━━ [jo_top_k sweep, reranker ON] cascaded — 5개 조합 ━━━
  [1/5] jo_top_k=10 ... -> MRR=0.5103 Recall@10=0.57 stage1_recall=0.55 (2.859초/쿼리)
  [2/5] jo_top_k=20 ... -> MRR=0.8529 Recall@10=0.94 stage1_recall=0.94 (2.06초/쿼리)
  [3/5] jo_top_k=30 ... -> MRR=0.8753 Recall@10=0.98 stage1_recall=0.97 (0.772초/쿼리)
  [4/5] jo_top_k=40 ... -> MRR=0.8753 Recall@10=0.98 stage1_recall=0.97 (0.39초/쿼리)
  [5/5] jo_top

In [23]:
import pandas as pd

df_stage4 = pd.read_csv("/content/eval_stage4_jotopk_cascaded.csv")
print("=== jo_top_k별 상세 결과 (reranker on/off, stage1_recall 포함) ===")
display(df_stage4)

df_final_updated = pd.read_csv("/content/eval_final_best.csv")
print("\n=== eval_final_best.csv (cascaded on/off 행 추가됨, 총 6행) ===")
display(df_final_updated)


=== jo_top_k별 상세 결과 (reranker on/off, stage1_recall 포함) ===


,variant,use_reranker,jo_top_k,alpha,rrf_k,fetch_k,rerank_k,use_cross_refs,max_cross_ref_tokens,Recall@1,...,FullCoverage@2,Recall@3,FullCoverage@3,Recall@5,FullCoverage@5,Recall@10,FullCoverage@10,MRR,stage1_recall,avg_sec_per_query
0,cascaded,True,30,0.3,20,30,30,True,NaN,0.79,...,0.76,0.96,0.81,0.97,0.83,0.98,0.90,0.8753,0.97,0.772
1,cascaded,True,50,0.3,20,30,30,True,NaN,0.79,...,0.76,0.96,0.81,0.97,0.83,0.98,0.89,0.8753,0.97,0.271
2,cascaded,True,40,0.3,20,30,30,True,NaN,0.79,...,0.76,0.96,0.81,0.97,0.83,0.98,0.89,0.8753,0.97,0.390
3,cascaded,True,20,0.3,20,30,30,True,NaN,0.78,...,0.74,0.93,0.79,0.93,0.82,0.94,0.89,0.8529,0.94,2.060
4,cascaded,True,10,0.3,20,30,30,True,NaN,0.46,...,0.45,0.56,0.48,0.57,0.49,0.57,0.54,0.5103,0.55,2.859
5,cascaded,False,20,1.0,20,20,10,True,NaN,0.70,...,0.64,0.84,0.69,0.92,0.78,0.97,0.89,0.7916,0.97,0.046
6,cascaded,False,40,1.0,20,20,10,True,NaN,0.70,...,0.64,0.84,0.69,0.92,0.78,0.97,0.87,0.7916,0.97,0.034
7,cascaded,False,30,1.0,20,20,10,True,NaN,0.70,...,0.64,0.84,0.69,0.92,0.78,0.97,0.87,0.7916,0.97,0.033
8,cascaded,False,50,1.0,20,20,10,True,NaN,0.70,...,0.64,0.84,0.69,0.92,0.78,0.97,0.87,0.7916,0.97,0.033
9,cascaded,False,10,1.0,20,20,10,True,NaN,0.69,...,0.64,0.85,0.71,0.91,0.78,0.96,0.94,0.7855,0.95,0.315



=== eval_final_best.csv (cascaded on/off 행 추가됨, 총 6행) ===


,variant,use_reranker,alpha,rrf_k,fetch_k,rerank_k,MRR,Recall@1,Recall@2,Recall@3,...,Recall@10,FullCoverage@1,FullCoverage@2,FullCoverage@3,FullCoverage@5,FullCoverage@10,use_cross_refs,max_cross_ref_tokens,jo_top_k,stage1_recall
0,jo,True,0.3,20,80,20,0.8843,0.80,0.95,0.97,...,0.98,0.59,0.77,0.83,0.84,0.89,NaN,NaN,NaN,NaN
1,jo,False,1.0,100,20,10,0.7800,0.68,0.82,0.86,...,0.95,0.53,0.68,0.72,0.77,0.84,NaN,NaN,NaN,NaN
2,ho,True,0.3,20,30,30,0.8850,0.80,0.95,0.97,...,0.98,0.59,0.77,0.82,0.83,0.89,True,NaN,NaN,NaN
3,ho,False,1.0,20,20,10,0.7820,0.69,0.81,0.83,...,0.95,0.51,0.65,0.70,0.75,0.81,True,NaN,NaN,NaN
4,cascaded,True,0.3,20,30,30,0.8753,0.79,0.94,0.96,...,0.98,0.58,0.76,0.81,0.83,0.90,True,NaN,30.0,0.97
5,cascaded,False,1.0,20,20,10,0.7916,0.70,0.81,0.84,...,0.97,0.52,0.64,0.69,0.78,0.89,True,NaN,20.0,0.97


## 9. 쿼리 단위 오류 분석 — 왜 이런 결과가 나왔고 무엇을 놓쳤는지

`eval_final_best.csv`에 확정된 jo/ho/cascaded 각각의 **reranker ON(프로덕션)**
설정으로 전체 쿼리를 다시 검색해서, 쿼리 단위로 아래를 자동 분류합니다.
(`load_variant_config(variant, use_reranker=True)`가 기본값이라, on/off 두 행
중 on 쪽만 골라서 봅니다 — off 쪽 오류 분석이 필요하면 `use_reranker=False`로
직접 호출하도록 스크립트를 고쳐 쓰면 됩니다.)

- **정확** / **순위_밀림** / **법령_혼동** / **완전_소실** / **카테고리_불명**
  (법령_혼동 vs 완전_소실은 `laws_ref.json`의 `category` 필드로 구분합니다 —
  1위로 올라온 오답이 정답과 다른 법령군인지 아닌지)
- **HoRAG(병렬) vs Cascaded(계층형) 회복/퇴행** — 세 variant 모두 GT를
  `relevant_docs_jo`(조 단위)로 통일했으므로 같은 정답 기준으로 직접 비교
  가능합니다. "ho는 놓쳤는데 cascaded가 살린 쿼리"와 그 반대를 각각 뽑아서,
  계층형 구조가 1단계 필터링 덕분에 실제로 무엇을 얻고 잃는지 사례로
  확인합니다.
- 필요하면 스크립트 안 `COMBOS`/`DIFF_PAIRS`를 채워서(예: `jo_top_k=30 vs 40`)
  기존처럼 두 조합 사이의 RR diff도 뽑을 수 있습니다 (기본은 비활성화 상태).

이 셀도 8번 섹션(cascaded sweep)이 먼저 끝나 있어야 `eval_final_best.csv`에
`cascaded` 행이 있어서 3-way 비교가 가능합니다. cascaded 행이 없으면 jo/ho만
비교하고 나머지는 자동으로 건너뜁니다.

**리랭커 자체가 얼마나 기여했는지(on vs off 지표 비교)는 여기가 아니라
10번 섹션의 비교표에서 확인하세요** — 이 섹션은 "왜 틀렸는지"를 사례로
파고드는 용도고, 10번은 "숫자가 얼마나 올랐는지"를 한눈에 보는 용도입니다.


In [15]:
%%writefile /content/yoonha_rag_query_diff.py
"""
Workit - 쿼리 단위 오류 분석 스크립트, reranker 단일화 버전 (jo/ho/cascaded 3-way)
파일명: rag/yoonha_rag_query_diff.py

목적:
  eval_final_best.csv / eval_stage*.csv는 조합별 집계 지표(Recall@k, MRR)만
  남기고 "쿼리 하나하나가 왜 맞았는지/틀렸는지"는 버린다. 이 스크립트는
  eval_final_best.csv에 확정된 jo/ho/cascaded 각각의 최종 설정으로 전체
  쿼리를 다시 검색해서, 쿼리 단위 원인 분석을 한다:

    1) 오답 유형 분류 (variant별로 각 쿼리를 아래 중 하나로 자동 태깅)
       - "정확"           : 정답이 1위
       - "순위_밀림"      : 정답이 top-k 안에는 있지만 1위가 아님
       - "법령_혼동"      : 정답을 top-k 안에 못 넣었고, 1위로 올라온 후보가
                            정답과 다른 category(법령군)에 속함
                            (예: 시행규칙이 정답인데 예규(PYG)가 1위로 올라옴 —
                            v3 문서 3-2a에서 확인된 패턴과 동일한 종류)
       - "완전_소실"      : 정답을 top-k 안에 못 넣었고, 1위 후보도 정답과
                            같은 category라 카테고리 문제는 아님 (세부 조항
                            자체를 못 찾은 경우)
       - "카테고리_불명"  : laws_ref에 category 정보가 없어 판단 불가
       (category는 laws_ref.json 기준이라, 여기 없는 chunk_id는 "불명" 처리됨)

    2) ho vs cascaded 회복/퇴행 비교
       - 세 variant 모두 GT를 relevant_docs_jo(조 단위)로 통일했으므로
         직접 비교 가능. "ho는 놓쳤는데 cascaded가 살린 쿼리"와 그 반대
         ("ho는 맞았는데 cascaded가 놓친 쿼리")를 각각 뽑아서 계층형 구조가
         실제로 뭘 얻고 뭘 잃는지 확인한다.

    3) (옵션) 임의의 두 조합(예: jo_top_k=30 vs 40) 사이 RR diff
       — COMBOS/DIFF_PAIRS를 고쳐서 기존처럼 쓸 수 있음.

출력:
  - query_level_results.csv     : jo/ho/cascaded × 전체 쿼리의 순위 결과 (long format)
  - error_analysis_summary.csv  : variant × 오류유형별 건수/비율
  - ho_vs_cascaded_recovered.csv / ho_vs_cascaded_regressed.csv
  - diff_<combo1>_vs_<combo2>.csv (DIFF_PAIRS가 비어있지 않을 때만)
  - 콘솔에 사람이 읽을 수 있는 요약 + 대표 사례 출력

사용법:
    python rag/yoonha_rag_query_diff.py
"""

from __future__ import annotations

import json
import os
from dataclasses import dataclass
from pathlib import Path

import pandas as pd
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue

from yoonha_law_rag import (
    load_embed_model,
    load_laws_ref,
    load_reranker,
    search_jo,
    search_ho,
    derive_jo_id,
    SweepCache,
    QDRANT_HOST,
    QDRANT_PORT,
    COLLECTION_HO,
)
from yoonha_law_rag_cascaded import search_cascaded

QDRANT_LOCAL_PATH_ENV = os.environ.get("QDRANT_LOCAL_PATH")
RERANKER_DEVICE = os.environ.get("RERANKER_DEVICE", "cpu")


def get_qdrant_client() -> QdrantClient:
    if QDRANT_LOCAL_PATH_ENV:
        print(f"🗄️  로컬(파일 기반) Qdrant 사용: {QDRANT_LOCAL_PATH_ENV}")
        return QdrantClient(path=QDRANT_LOCAL_PATH_ENV)
    print(f"🗄️  서버 Qdrant 사용: {QDRANT_HOST}:{QDRANT_PORT}")
    return QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)


GOLD_STANDARD_PATH = Path("/content/eval_set.json")
OUT_DIR = Path("/content")
FINAL_BEST_PATH = OUT_DIR / "eval_final_best.csv"
CACHE_PATH = OUT_DIR / "sweep_cache_query_diff.pkl"  # 세션 끊겨도 이어받을 수 있게 디스크에 저장

TOP_K_EVAL = 10

# LLM에는 최종적으로 조 텍스트가 전달되므로(HoRAG/Cascaded도 parent fetch로
# 조 텍스트를 붙여서 반환), 세 variant 모두 GT를 relevant_docs_jo로 통일한다.
GT_FIELD_BY_VARIANT = {
    "jo": "relevant_docs_jo",
    "ho": "relevant_docs_jo",
    "cascaded": "relevant_docs_jo",
}

# ho/cascaded는 raw chunk_id가 호/목/세목 단위라, 넉넉히 뽑아 조 단위로 접은
# 뒤 순서를 유지한 채 중복 제거한다 (jo/ho eval 스크립트와 동일 로직).
RAW_TOP_K_FOR_JO_DEDUP = 30


def load_gold_standard(path: Path = GOLD_STANDARD_PATH) -> list[dict]:
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def load_variant_config(variant: str, use_reranker: bool = True) -> dict | None:
    """
    eval_final_best.csv에서 해당 variant + use_reranker 조합의 확정 설정을 읽어
    search_*에 그대로 넘길 kwargs로 변환한다. 행이 없으면 None (예: cascaded
    sweep을 아직 안 돌렸을 때). eval_final_best.csv는 이제 variant마다 on/off
    2행씩 있으므로 use_reranker로 반드시 구분해서 골라야 한다 — 기본값 True는
    "오류 분석은 프로덕션 후보(reranker ON) 기준으로 본다"는 의도다.
    """
    if not FINAL_BEST_PATH.exists():
        raise FileNotFoundError(f"{FINAL_BEST_PATH} 가 없습니다. 먼저 sweep 스크립트를 실행하세요.")
    df = pd.read_csv(FINAL_BEST_PATH)
    rows = df[(df["variant"] == variant) & (df["use_reranker"] == use_reranker)]
    if rows.empty:
        return None
    row = rows.iloc[0]

    cfg = dict(
        alpha=float(row["alpha"]),
        rrf_k=int(row["rrf_k"]),
        fetch_k=int(row["fetch_k"]),
        rerank_k=int(row["rerank_k"]),
        use_reranker=use_reranker,
    )
    if variant in ("ho", "cascaded"):
        max_tok = row.get("max_cross_ref_tokens", None)
        cfg["use_cross_refs"] = bool(row["use_cross_refs"])
        cfg["max_cross_ref_tokens"] = None if pd.isna(max_tok) else int(max_tok)
    if variant == "cascaded":
        cfg["jo_top_k"] = int(row["jo_top_k"])
    return cfg


def _lookup_text(chunk_id: str, client: QdrantClient, cache: SweepCache, snippet_len: int = 70) -> str:
    """chunk_id의 실제 조문 텍스트 앞부분을 조회 (diff/사례 출력용)."""
    cache_key = (COLLECTION_HO, chunk_id)
    if cache_key in cache.scroll:
        payload = cache.scroll[cache_key]
    else:
        results = client.scroll(
            collection_name=COLLECTION_HO,
            scroll_filter=Filter(must=[FieldCondition(key="chunk_id", match=MatchValue(value=chunk_id))]),
            limit=1, with_payload=True, with_vectors=False,
        )
        points = results[0]
        if not points:
            return "(텍스트 없음)"
        payload = points[0].payload
        cache.scroll[cache_key] = payload

    text = payload.get("text", payload.get("chunk_text", ""))
    return text[:snippet_len].replace("\n", " ") + ("…" if len(text) > snippet_len else "")


def classify_error(rank_of_hit: int | None, gt_docs: set[str], top1_chunk_id: str | None,
                    laws_ref: dict) -> str:
    """오답 유형 자동 분류. laws_ref 기준 category 비교로 '법령 혼동' vs '완전 소실'을 나눈다."""
    if rank_of_hit == 1:
        return "정확"
    if rank_of_hit is not None:
        return "순위_밀림"

    gt_categories = {laws_ref.get(d, {}).get("category", "") for d in gt_docs}
    gt_categories = {c for c in gt_categories if c}
    top1_category = laws_ref.get(top1_chunk_id, {}).get("category", "") if top1_chunk_id else ""

    if not gt_categories or not top1_category:
        return "카테고리_불명"
    if top1_category not in gt_categories:
        return "법령_혼동"
    return "완전_소실"


def run_variant(variant: str, cfg: dict, client, model, laws_ref, reranker,
                 gold_standard: list[dict], cache: SweepCache) -> list[dict]:
    """variant(jo/ho/cascaded) 하나에 대해 전체 쿼리의 순위 결과 + 오류유형을 반환."""
    search_fn = {"jo": search_jo, "ho": search_ho, "cascaded": search_cascaded}[variant]
    gt_field = GT_FIELD_BY_VARIANT[variant]
    rows = []

    for item in gold_standard:
        gt_docs = set(item[gt_field])
        query = item["query"]

        law_refs = search_fn(
            clause_text=query, client=client, model=model, laws_ref=laws_ref,
            reranker=reranker, top_k=RAW_TOP_K_FOR_JO_DEDUP, cache=cache, **cfg,
        )
        raw_ranked_ids = [r.chunk_id for r in law_refs]

        # 조 id로 접고, 순서를 유지한 채 중복만 제거 (jo variant는 사실상 no-op)
        seen: set[str] = set()
        ranked_ids: list[str] = []
        for cid in raw_ranked_ids:
            jid = derive_jo_id(cid)
            if jid not in seen:
                seen.add(jid)
                ranked_ids.append(jid)

        rank_of_hit = None
        for rank, cid in enumerate(ranked_ids, 1):
            if cid in gt_docs:
                rank_of_hit = rank
                break
        rr = round(1 / rank_of_hit, 4) if rank_of_hit else 0.0
        top1 = ranked_ids[0] if ranked_ids else None

        rows.append({
            "variant": variant,
            "query_id": item.get("query_id", ""),
            "query": query,
            "gt_docs": list(gt_docs),
            "rank_of_hit": rank_of_hit,
            "rr": rr,
            "hit@1": bool(rank_of_hit == 1),
            "hit@5": bool(rank_of_hit is not None and rank_of_hit <= 5),
            "hit@10": bool(rank_of_hit is not None and rank_of_hit <= 10),
            "ranked_ids_top5": ranked_ids[:5],
            "error_type": classify_error(rank_of_hit, gt_docs, top1, laws_ref),
        })

    return rows


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# (옵션) 임의의 두 조합끼리 RR diff — 필요할 때만 COMBOS/DIFF_PAIRS를 채워서 사용
# 예: jo_top_k를 30 vs 40으로 바꿔가며 비교하고 싶을 때
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

@dataclass
class ComboSpec:
    name: str
    variant: str  # "jo" | "ho" | "cascaded"
    overrides: dict  # base config에 덮어쓸 파라미터 (예: {"jo_top_k": 30})


COMBOS: list[ComboSpec] = [
    # 예시 (필요하면 주석 해제 후 jo_top_k 값을 바꿔서 비교):
    # ComboSpec("cascaded_k30", "cascaded", {"jo_top_k": 30}),
    # ComboSpec("cascaded_k40", "cascaded", {"jo_top_k": 40}),
]
DIFF_PAIRS: list[tuple[str, str]] = [
    # ("cascaded_k30", "cascaded_k40"),
]


def print_and_save_diff(df: pd.DataFrame, combo1: str, combo2: str, client, cache) -> pd.DataFrame:
    a = df[df["variant"] == combo1].set_index("query_id")
    b = df[df["variant"] == combo2].set_index("query_id")
    merged = a[["query", "gt_docs", "rr", "rank_of_hit", "ranked_ids_top5"]].join(
        b[["rr", "rank_of_hit", "ranked_ids_top5"]], lsuffix="_1", rsuffix="_2"
    )
    merged["rr_delta"] = merged["rr_2"] - merged["rr_1"]
    changed = merged[merged["rr_delta"] != 0].sort_values("rr_delta")

    print(f"\n{'='*70}\n  {combo1}  vs  {combo2}\n  총 {len(merged)}개 쿼리 중 {len(changed)}개 쿼리에서 순위 변화\n{'='*70}")
    for qid, row in changed.iterrows():
        direction = "🔺 개선" if row["rr_delta"] > 0 else "🔻 악화"
        print(f"\n[{qid}] {direction}  RR: {row['rr_1']} → {row['rr_2']}")
        print(f"  질의: {row['query'][:80]}")
        print(f"  정답: {row['gt_docs']}")
        for label, col in [(combo1, "ranked_ids_top5_1"), (combo2, "ranked_ids_top5_2")]:
            print(f"  [{label}] top3:")
            for cid in row[col][:3]:
                mark = "✅" if cid in row["gt_docs"] else "  "
                print(f"    {mark} {cid} — {_lookup_text(cid, client, cache)}")

    out_path = OUT_DIR / f"diff_{combo1}_vs_{combo2}.csv"
    changed.reset_index().to_csv(out_path, index=False)
    print(f"\n✅ diff 저장: {out_path}")
    return changed


def print_error_summary(df: pd.DataFrame) -> pd.DataFrame:
    summary = (
        df.groupby(["variant", "error_type"]).size().rename("count").reset_index()
    )
    summary["ratio"] = summary["count"] / summary.groupby("variant")["count"].transform("sum")
    summary["ratio"] = summary["ratio"].round(3)
    print("\n" + "=" * 60)
    print("오답 유형별 집계 (variant × error_type)")
    print("=" * 60)
    print(summary.to_string(index=False))
    summary.to_csv(OUT_DIR / "error_analysis_summary.csv", index=False)
    print(f"\n✅ 저장: {OUT_DIR / 'error_analysis_summary.csv'}")
    return summary


def print_ho_vs_cascaded(df: pd.DataFrame, client, cache) -> None:
    """ho와 cascaded는 같은 GT(relevant_docs_jo) 기준이라 직접 회복/퇴행 비교가 가능."""
    ho = df[df["variant"] == "ho"].set_index("query_id")
    casc = df[df["variant"] == "cascaded"]
    if casc.empty:
        print("\n(cascaded 결과 없음 — jo_top_k sweep을 먼저 실행하세요)")
        return
    casc = casc.set_index("query_id")

    joined = ho[["query", "gt_docs", "rank_of_hit", "ranked_ids_top5"]].join(
        casc[["rank_of_hit", "ranked_ids_top5"]], lsuffix="_ho", rsuffix="_casc"
    )

    recovered = joined[(joined["rank_of_hit_ho"].isna()) & (joined["rank_of_hit_casc"].notna())]
    regressed = joined[(joined["rank_of_hit_ho"].notna()) & (joined["rank_of_hit_casc"].isna())]

    print("\n" + "=" * 60)
    print(f"HoRAG(병렬) vs Cascaded(계층형) 회복/퇴행 — 총 {len(joined)}개 쿼리")
    print(f"  🔺 회복(ho는 놓침 → cascaded는 맞춤): {len(recovered)}개")
    print(f"  🔻 퇴행(ho는 맞춤 → cascaded는 놓침): {len(regressed)}개")
    print("=" * 60)

    for label, sub in [("회복", recovered), ("퇴행", regressed)]:
        for qid, row in sub.iterrows():
            print(f"\n[{qid}] {label}")
            print(f"  질의: {row['query'][:80]}")
            print(f"  정답: {row['gt_docs']}")
            print(f"  [ho]       top3: {row['ranked_ids_top5_ho'][:3]}")
            print(f"  [cascaded] top3: {row['ranked_ids_top5_casc'][:3]}")

    recovered.reset_index().to_csv(OUT_DIR / "ho_vs_cascaded_recovered.csv", index=False)
    regressed.reset_index().to_csv(OUT_DIR / "ho_vs_cascaded_regressed.csv", index=False)
    print(f"\n✅ 저장: ho_vs_cascaded_recovered.csv / ho_vs_cascaded_regressed.csv")


def main():
    client = get_qdrant_client()
    model = load_embed_model()
    laws_ref = load_laws_ref()
    gold_standard = load_gold_standard()
    print(f"실버 스탠다드 {len(gold_standard)}개 로드 완료")

    reranker = load_reranker(device=RERANKER_DEVICE)
    cache = SweepCache.load_or_new(CACHE_PATH)

    all_rows: list[dict] = []

    try:
        # 1) jo/ho/cascaded — eval_final_best.csv에 확정된 최종 설정 그대로 재실행
        for variant in ("jo", "ho", "cascaded"):
            cfg = load_variant_config(variant)
            if cfg is None:
                print(f"\n(variant='{variant}' 확정 설정이 eval_final_best.csv에 없어 건너뜁니다)")
                continue
            print(f"\n▶ {variant} 실행 중 (설정: {cfg})")
            all_rows.extend(run_variant(variant, cfg, client, model, laws_ref, reranker, gold_standard, cache))
            cache.save(CACHE_PATH)  # variant 하나 끝날 때마다 저장

        # 2) (옵션) COMBOS에 정의된 임의 조합들도 같이 실행
        for combo in COMBOS:
            base_cfg = load_variant_config(combo.variant) or {}
            cfg = {**base_cfg, **combo.overrides}
            print(f"\n▶ {combo.name} 실행 중 ({combo.variant}, overrides={combo.overrides})")
            rows = run_variant(combo.variant, cfg, client, model, laws_ref, reranker, gold_standard, cache)
            for r in rows:
                r["variant"] = combo.name  # DIFF_PAIRS에서 조합 이름으로 구분하기 위해 덮어씀
            all_rows.extend(rows)
    finally:
        cache.save(CACHE_PATH)

    df = pd.DataFrame(all_rows)
    df_to_save = df.copy()
    df_to_save["gt_docs"] = df_to_save["gt_docs"].apply(json.dumps, ensure_ascii=False)
    df_to_save["ranked_ids_top5"] = df_to_save["ranked_ids_top5"].apply(json.dumps, ensure_ascii=False)
    df_to_save.to_csv(OUT_DIR / "query_level_results.csv", index=False)
    print(f"\n✅ 쿼리 단위 전체 결과 저장: {OUT_DIR / 'query_level_results.csv'}")

    print_error_summary(df)
    print_ho_vs_cascaded(df, client, cache)

    for combo1, combo2 in DIFF_PAIRS:
        print_and_save_diff(df, combo1, combo2, client, cache)


if __name__ == "__main__":
    main()


Writing /content/yoonha_rag_query_diff.py


In [16]:
import os

os.environ["QDRANT_LOCAL_PATH"] = "/content/qdrant_local"
os.environ["RERANKER_DEVICE"] = "cuda"

!python /content/yoonha_rag_query_diff.py


🗄️  로컬(파일 기반) Qdrant 사용: /content/qdrant_local
📦 임베딩 모델 로드: BAAI/bge-m3
Fetching 30 files: 100% 30/30 [00:00<00:00, 11607.85it/s]
Download complete: : 0.00B [00:00, ?B/s]
Loading weights: 100% 391/391 [00:09<00:00, 42.92it/s]
  ⚠️  laws_ref.json 없음: /data/hn_seed/law_refs.json
실버 스탠다드 100개 로드 완료
📦 Re-ranker 로드: BAAI/bge-reranker-v2-m3
Loading weights: 100% 393/393 [00:00<00:00, 863.20it/s]
  💾 캐시 파일 없음, 새로 시작: /content/sweep_cache_query_diff.pkl

▶ jo 실행 중 (설정: {'alpha': 0.3, 'rrf_k': 20, 'fetch_k': 80, 'rerank_k': 20, 'use_reranker': True})
  💾 캐시 저장: /content/sweep_cache_query_diff.pkl ({'embed_cached': 100, 'raw_search_cached': 100, 'scroll_cached': 0, 'rerank_cached': 10575})

▶ ho 실행 중 (설정: {'alpha': 0.3, 'rrf_k': 20, 'fetch_k': 30, 'rerank_k': 30, 'use_reranker': True, 'use_cross_refs': True, 'max_cross_ref_tokens': None})
  💾 캐시 저장: /content/sweep_cache_query_diff.pkl ({'embed_cached': 100, 'raw_search_cached': 200, 'scroll_cached': 1994, 'rerank_cached': 18253})

▶ cascaded 실행 

In [24]:
import pandas as pd

df_errors = pd.read_csv("/content/error_analysis_summary.csv")
print("=== 오답 유형별 집계 ===")
display(df_errors)

df_query = pd.read_csv("/content/query_level_results.csv")
print("\n=== 쿼리 단위 전체 결과 (long format) ===")
display(df_query.head(20))


=== 오답 유형별 집계 ===


,variant,error_type,count,ratio
0,cascaded,순위_밀림,19,0.19
1,cascaded,정확,79,0.79
2,cascaded,카테고리_불명,2,0.02
3,ho,순위_밀림,19,0.19
4,ho,정확,80,0.80
5,ho,카테고리_불명,1,0.01
6,jo,순위_밀림,19,0.19
7,jo,정확,80,0.80
8,jo,카테고리_불명,1,0.01



=== 쿼리 단위 전체 결과 (long format) ===


,variant,query_id,query,gt_docs,rank_of_hit,rr,hit@1,hit@5,hit@10,ranked_ids_top5,error_type
0,jo,v01,A건설사와 B건설사가 사전에 서로 상의하여 입찰가격과 낙찰자를 미리 정해놓고 입찰에...,"[""LCA_0_0_31""]",1.0,1.0000,True,True,True,"[""LCA_0_0_31"", ""PYG_9_1_0_1"", ""LCA_0_0_13"", ""L...",정확
1,jo,v02,한 기업이 가명처리한 고객 데이터를 이용해 특정 개인을 다시 알아보려는 목적으로 재...,"[""PIPA_3_3_28-5"", ""PIPA_3_3_28-3"", ""PIPA_3_3_2...",1.0,1.0000,True,True,True,"[""PIPA_3_3_28-5"", ""PIPA_9_0_64-2"", ""PIPA_10_0_...",정확
2,jo,v03,개인정보처리자가 고객 정보 유출 사실을 인지하고도 5일이 지난 뒤에야 정보주체에게 ...,"[""PIPAE_5_0_39"", ""PIPA_4_0_34"", ""PIPAE_5_0_40""]",1.0,1.0000,True,True,True,"[""PIPAE_5_0_39"", ""PIPA_4_0_34"", ""PIPAE_4_0_15-...",정확
3,jo,v04,"부정당업자 요건에 해당하는 업체가 과징금 부과 요건도 함께 충족하는데, 지방자치단체...","[""LCA_0_0_31"", ""LCA_0_0_31-2""]",1.0,1.0000,True,True,True,"[""LCA_0_0_31-2"", ""LCAE_7_0_111"", ""LCA_0_0_31"",...",정확
4,jo,v05,한 업체가 대부계약 기간이 끝난 뒤에도 별도 계약 없이 계속 공유재산을 사용하고 있...,"[""PPMA_1_0_2""]",1.0,1.0000,True,True,True,"[""PPMA_1_0_2"", ""PPMA_8_0_81"", ""PPMA_8_0_80"", ""...",정확
5,jo,v06,계약상대자가 계약금액의 100분의 5에 해당하는 금액만 계약보증금으로 납부했습니다....,"[""LCAE_5_0_51""]",3.0,0.3333,False,True,True,"[""PYG_9_4_0_1"", ""LCA_0_0_15"", ""LCAE_5_0_51"", ""...",순위_밀림
6,jo,v07,발주기관이 소프트웨어사업 계약서에 계약상대자에게 현저히 불리한 조항을 일방적으로 포...,"[""SWPA_5_1_38""]",1.0,1.0000,True,True,True,"[""SWPA_5_1_38"", ""SWPAE_5_1_30"", ""PYG_9_1_0_1"",...",정확
7,jo,v08,하도급대금 직불조건부 입찰참가 통보를 받은 업체가 통보일로부터 6개월 만에 별다른 ...,"[""LCA_0_0_18"", ""LCA_0_0_31-4""]",1.0,1.0000,True,True,True,"[""LCA_0_0_31-4"", ""LCAE_3_0_19"", ""LCAR_3_0_17"",...",정확
8,jo,v09,지방자치단체가 회계연도가 끝나는 날이 아니라 그 다음 달 말일에 출납을 폐쇄했습니다...,"[""LARA_1_0_7""]",1.0,1.0000,True,True,True,"[""LARA_1_0_7"", ""LARAE_1_0_3"", ""LARA_1_0_6"", ""L...",정확
9,jo,v10,공사 원가계산 시 시행령 제10조제1항제2호에 따른 절차를 따르지 않고 담당자가 임...,"[""LCAE_2_0_10"", ""LCAR_2_0_6""]",1.0,1.0000,True,True,True,"[""LCAE_2_0_10"", ""LCAR_2_0_6"", ""LCAR_2_0_5"", ""L...",정확


## 10. reranker ON vs OFF — 지표 비교표

jo/ho/cascaded 각각에 대해 리랭커를 켰을 때/껐을 때 지표가 얼마나 달라지는지
한눈에 보는 비교표입니다. Qdrant/임베딩/리랭커를 전혀 로드하지 않는 순수
후처리라 즉시 실행됩니다 (GPU 불필요) — `eval_final_best.csv`만 있으면 됩니다.

5번(jo/ho on/off)까지만 끝나도 두 variant는 비교되고, 8번(cascaded on/off)
까지 끝나면 세 variant 전부 비교됩니다. 아직 안 돌린 조합은 "미실행"으로
표시됩니다.


In [17]:
%%writefile /content/yoonha_rag_reranker_impact.py
"""
Workit - reranker on/off 비교표
파일명: rag/yoonha_rag_reranker_impact.py

목적:
  eval_final_best.csv에 쌓인 jo/ho/cascaded × reranker on/off 6행을 가지고,
  "리랭커를 켜면 지표가 얼마나 올라가는지"를 variant별로 한눈에 보이는
  비교표로 정리한다. Qdrant/임베딩/리랭커를 전혀 로드하지 않는 순수 후처리
  스크립트라 이 셀은 즉시 실행된다 (GPU 불필요).

  8번(cascaded sweep)까지 다 끝나야 6행이 채워진다. jo/ho 2개(4행)만 있는
  상태에서 먼저 돌려도 동작은 하고, cascaded 행이 없으면 그 부분만 "미실행"
  으로 표시한다.

출력:
  - eval_reranker_on_off_comparison.csv
  - 콘솔에 variant별 MRR/Recall/FullCoverage on-off 비교 + delta 출력
"""

from __future__ import annotations

from pathlib import Path

import pandas as pd

OUT_DIR = Path("/content")
FINAL_BEST_PATH = OUT_DIR / "eval_final_best.csv"

METRICS = ["MRR", "Recall@1", "Recall@5", "Recall@10", "FullCoverage@5", "FullCoverage@10"]
VARIANT_LABELS = {"jo": "JoRAG", "ho": "HoRAG(병렬)", "cascaded": "계층형(Cascaded)"}


def main():
    if not FINAL_BEST_PATH.exists():
        raise FileNotFoundError(f"{FINAL_BEST_PATH} 가 없습니다. sweep 스크립트를 먼저 실행하세요.")

    df = pd.read_csv(FINAL_BEST_PATH)
    metrics = [m for m in METRICS if m in df.columns]

    rows = []
    for variant in ["jo", "ho", "cascaded"]:
        sub = df[df["variant"] == variant]
        on_row = sub[sub["use_reranker"] == True]
        off_row = sub[sub["use_reranker"] == False]

        if on_row.empty and off_row.empty:
            continue

        row = {"variant": variant, "label": VARIANT_LABELS.get(variant, variant)}
        for m in metrics:
            on_val = float(on_row.iloc[0][m]) if not on_row.empty else None
            off_val = float(off_row.iloc[0][m]) if not off_row.empty else None
            row[f"{m}_off"] = off_val
            row[f"{m}_on"] = on_val
            row[f"{m}_delta"] = (round(on_val - off_val, 4)
                                  if (on_val is not None and off_val is not None) else None)
        rows.append(row)

    df_compare = pd.DataFrame(rows)
    df_compare.to_csv(OUT_DIR / "eval_reranker_on_off_comparison.csv", index=False)

    print("=" * 70)
    print("reranker ON vs OFF — variant별 지표 비교 (delta = ON - OFF)")
    print("=" * 70)

    for row in rows:
        print(f"\n■ {row['label']} ({row['variant']})")
        for m in metrics:
            off_val, on_val, delta = row[f"{m}_off"], row[f"{m}_on"], row[f"{m}_delta"]
            if off_val is None or on_val is None:
                status = "미실행 (해당 조합 없음)"
                print(f"    {m:16s}: {status}")
                continue
            arrow = "▲" if delta > 0 else ("▼" if delta < 0 else "―")
            print(f"    {m:16s}: {off_val:.4f} (OFF) -> {on_val:.4f} (ON)   {arrow} {delta:+.4f}")

    missing = {"jo", "ho", "cascaded"} - {r["variant"] for r in rows}
    if missing:
        print(f"\n(참고: {sorted(missing)} 행이 아직 eval_final_best.csv에 없어 비교에서 제외됨)")

    print(f"\n✅ 저장: {OUT_DIR / 'eval_reranker_on_off_comparison.csv'}")


if __name__ == "__main__":
    main()


Writing /content/yoonha_rag_reranker_impact.py


In [18]:
!python /content/yoonha_rag_reranker_impact.py


reranker ON vs OFF — variant별 지표 비교 (delta = ON - OFF)

■ JoRAG (jo)
    MRR             : 0.7800 (OFF) -> 0.8843 (ON)   ▲ +0.1043
    Recall@1        : 0.6800 (OFF) -> 0.8000 (ON)   ▲ +0.1200
    Recall@5        : 0.8800 (OFF) -> 0.9800 (ON)   ▲ +0.1000
    Recall@10       : 0.9500 (OFF) -> 0.9800 (ON)   ▲ +0.0300
    FullCoverage@5  : 0.7700 (OFF) -> 0.8400 (ON)   ▲ +0.0700
    FullCoverage@10 : 0.8400 (OFF) -> 0.8900 (ON)   ▲ +0.0500

■ HoRAG(병렬) (ho)
    MRR             : 0.7820 (OFF) -> 0.8850 (ON)   ▲ +0.1030
    Recall@1        : 0.6900 (OFF) -> 0.8000 (ON)   ▲ +0.1100
    Recall@5        : 0.8900 (OFF) -> 0.9800 (ON)   ▲ +0.0900
    Recall@10       : 0.9500 (OFF) -> 0.9800 (ON)   ▲ +0.0300
    FullCoverage@5  : 0.7500 (OFF) -> 0.8300 (ON)   ▲ +0.0800
    FullCoverage@10 : 0.8100 (OFF) -> 0.8900 (ON)   ▲ +0.0800

■ 계층형(Cascaded) (cascaded)
    MRR             : 0.7916 (OFF) -> 0.8753 (ON)   ▲ +0.0837
    Recall@1        : 0.7000 (OFF) -> 0.7900 (ON)   ▲ +0.0900
    Recall@5     

In [19]:
import pandas as pd

df_impact = pd.read_csv("/content/eval_reranker_on_off_comparison.csv")
display(df_impact)


,variant,label,MRR_off,MRR_on,MRR_delta,Recall@1_off,Recall@1_on,Recall@1_delta,Recall@5_off,Recall@5_on,Recall@5_delta,Recall@10_off,Recall@10_on,Recall@10_delta,FullCoverage@5_off,FullCoverage@5_on,FullCoverage@5_delta,FullCoverage@10_off,FullCoverage@10_on,FullCoverage@10_delta
0,jo,JoRAG,0.7800,0.8843,0.1043,0.68,0.80,0.12,0.88,0.98,0.10,0.95,0.98,0.03,0.77,0.84,0.07,0.84,0.89,0.05
1,ho,HoRAG(병렬),0.7820,0.8850,0.1030,0.69,0.80,0.11,0.89,0.98,0.09,0.95,0.98,0.03,0.75,0.83,0.08,0.81,0.89,0.08
2,cascaded,계층형(Cascaded),0.7916,0.8753,0.0837,0.70,0.79,0.09,0.92,0.97,0.05,0.97,0.98,0.01,0.78,0.83,0.05,0.89,0.90,0.01


## 11. 결과 통합 다운로드

위 5~10번 섹션을 원하는 만큼 돌린 뒤, 이 셀을 마지막에 한 번만 실행하세요.


In [25]:
import pandas as pd
from pathlib import Path
from google.colab import files

OUT_DIR = Path("/content")


def read_csv_or_none(name: str) -> pd.DataFrame | None:
    p = OUT_DIR / name
    return pd.read_csv(p) if p.exists() else None


def merge_variant_reranker(file_map: dict[tuple[str, bool], str]) -> pd.DataFrame | None:
    """{(variant, use_reranker): 파일명} 을 컬럼 추가해서 하나로 합친다."""
    frames = []
    for (variant, use_reranker), fname in file_map.items():
        df = read_csv_or_none(fname)
        if df is None:
            continue
        df = df.copy()
        df["variant"] = variant
        df["use_reranker"] = use_reranker
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else None


# stage1: alpha x rrf_k (jo/ho, on/off) — 4개 파일 -> 1개 시트
stage1_all = merge_variant_reranker({
    ("jo", True):  "eval_stage1_alpha_rrf_jo.csv",
    ("jo", False): "eval_stage1_off_alpha_rrf_jo.csv",
    ("ho", True):  "eval_stage1_alpha_rrf_ho.csv",
    ("ho", False): "eval_stage1_off_alpha_rrf_ho.csv",
})

# stage2: fetch_k x rerank_k (jo/ho, on/off) — 4개 파일 -> 1개 시트
stage2_all = merge_variant_reranker({
    ("jo", True):  "eval_stage2_fetchk_rerankk_jo.csv",
    ("jo", False): "eval_stage2_off_fetchk_jo.csv",
    ("ho", True):  "eval_stage2_fetchk_rerankk_ho.csv",
    ("ho", False): "eval_stage2_off_fetchk_ho.csv",
})

# stage3: cross_refs on/off (ho 전용) — 2개 파일 -> 1개 시트
stage3_all = merge_variant_reranker({
    ("ho", True):  "eval_stage3_crossref_ho.csv",
    ("ho", False): "eval_stage3_off_crossref_ho.csv",
})

# ho_vs_cascaded recovered/regressed — direction 컬럼으로 병합 (있을 때만)
diff_frames = []
recovered = read_csv_or_none("ho_vs_cascaded_recovered.csv")
if recovered is not None:
    recovered = recovered.copy()
    recovered["direction"] = "recovered"
    diff_frames.append(recovered)
regressed = read_csv_or_none("ho_vs_cascaded_regressed.csv")
if regressed is not None:
    regressed = regressed.copy()
    regressed["direction"] = "regressed"
    diff_frames.append(regressed)
ho_vs_cascaded_diff = pd.concat(diff_frames, ignore_index=True) if diff_frames else None

# 그대로 시트로 들어갈 것들
final_best = read_csv_or_none("eval_final_best.csv")
stage4_cascaded = read_csv_or_none("eval_stage4_jotopk_cascaded.csv")
query_level = read_csv_or_none("query_level_results.csv")
error_summary = read_csv_or_none("error_analysis_summary.csv")
reranker_compare = read_csv_or_none("eval_reranker_on_off_comparison.csv")

sheets = {
    "stage1_alpha_rrf": stage1_all,
    "stage2_fetchk_rerankk": stage2_all,
    "stage3_crossref_ho": stage3_all,
    "final_best": final_best,
    "stage4_jotopk_cascaded": stage4_cascaded,
    "query_level_results": query_level,
    "error_analysis_summary": error_summary,
    "ho_vs_cascaded_diff": ho_vs_cascaded_diff,
    "reranker_on_off_comparison": reranker_compare,
}

output_path = OUT_DIR / "workit_rag_eval_results.xlsx"
written = []
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for sheet_name, df in sheets.items():
        if df is None:
            continue
        # 엑셀 시트명은 31자 제한
        df.to_excel(writer, sheet_name=sheet_name[:31], index=False)
        written.append(sheet_name)

missing = [name for name, df in sheets.items() if df is None]
print(f"✅ {len(written)}개 시트로 통합 완료: {written}")
if missing:
    print(f"⚠️  해당 CSV가 없어서 제외된 시트: {missing} (아직 안 돌린 섹션이 있으면 정상)")
print(f"\n📦 {output_path} 하나만 다운로드합니다 (기존 최대 17개 → 1개).")

files.download(str(output_path))


/tmp/ipykernel_367/948819784.py:61: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  ho_vs_cascaded_diff = pd.concat(diff_frames, ignore_index=True) if diff_frames else None


✅ 9개 시트로 통합 완료: ['stage1_alpha_rrf', 'stage2_fetchk_rerankk', 'stage3_crossref_ho', 'final_best', 'stage4_jotopk_cascaded', 'query_level_results', 'error_analysis_summary', 'ho_vs_cascaded_diff', 'reranker_on_off_comparison']

📦 /content/workit_rag_eval_results.xlsx 하나만 다운로드합니다 (기존 최대 17개 → 1개).


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>